In [ ]:
import joblib
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.config import load_paths
from model.model import read_scale_data
from evaluation.flux_energy import physical_values
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np

In [ ]:
#### insert data set
metrics_dir = '/work/bb1097/b383521/ML_models/coarsed_grained/metrics/'
metrics_data = joblib.load(metrics_dir + 'results_cnn_coarsed_grained_2.pkl')
regimes      = joblib.load(metrics_dir + 'regimes_cnn_coarsed_grained_2.pkl')
tag = ["land", "ocean"]
season_tag = ['jan', 'aug', 'combined']
metrics = ['mse_train', 'mse_val', 'mse_test', 'R2_train', 'R2_val', 'R2_test', 'mae_train', 'mae_val', 'mae_test', 'mae_train_norm_std', 
           'mae_val_norm_std', 'mae_test_norm_std', 'skill_train', 'skill_val', 'skill_test', 'n_train', 'n_val', 'n_test']

In [ ]:
import numpy as np
import pandas as pd

rows = []
index = []

for t in tag:
    for s in season_tag:
        row = {}
        for m in metrics:
            try:
                val = metrics_data[t][s][m]
                row[m] = np.nanmean(val)   # collapse the 144 values into their mean
            except KeyError:
                row[m] = np.nan
        rows.append(row)
        index.append((t, s))

df = pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(index, names=["tag", "season"]))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print(df.round(4))

In [ ]:
paths = load_paths()
save_dir = paths["save_dir"]
(_, _, y_train_land, _, _, _, _, ntri_land, _, _, _, 
         _, _, _, _, _, _) = read_scale_data(
            model_save_dir=save_dir,
            coarse_action_dir_jan=paths["coarse_action_dir_jan"],
            land_ocean_tag="land"
        )
(_, _, y_train_ocean, _, _, _, _, ntri_ocean, _, _, _, 
         _, _, _, _, _, _) = read_scale_data(
            model_save_dir=save_dir,
            coarse_action_dir_jan=paths["coarse_action_dir_jan"],
            land_ocean_tag="ocean"
        )

In [ ]:
with  xr.open_dataset(f"/work/bb1097/b383521/GW_Data/coarsed_action/coarse_gw_spectrum_group_{1}_fileslide_0.nc") as ds:
      k  = ds.k.data
      l  = ds.l.data
      m  = ds.m.data
# after re-running/restoring the original k, l, m wavenumber arrays:
k_wavenumber_array = k    # or whatever the correct array is, copy it to a safer name
l_wavenumber_array = l

In [ ]:
k_wavenumber_array

In [ ]:
import numpy as np

N_train_land = np.expm1(y_train_land)
N_train_ocean = np.expm1(y_train_ocean)
N_ing_train_land = np.sum(N_train_land, axis=1)
N_ing_train_ocean = np.sum(N_train_ocean, axis=1)

idx_max_ocean = np.argmax(N_ing_train_ocean)
idx_max_land = np.argmax(N_ing_train_land)

max_y_train_land = y_train_land[idx_max_land, :].reshape(6, 6, 4)
max_y_train_ocean = y_train_ocean[idx_max_ocean, :].reshape(6, 6, 4)

max_integrated_land = np.sum(np.expm1(max_y_train_land))
max_integrated_ocean = np.sum(np.expm1(max_y_train_ocean))

mean_y_train_land = np.nanmean(y_train_land, axis=0).reshape(6, 6, 4)
mean_y_train_ocean = np.nanmean(y_train_ocean, axis=0).reshape(6, 6, 4)
std_y_train_land = np.nanstd(y_train_land, axis=0).reshape(6, 6, 4)
std_y_train_ocean = np.nanstd(y_train_ocean, axis=0).reshape(6, 6, 4)

#mean_integrated_land = np.sum(np.expm1(mean_y_train_land))
#mean_integrated_ocean = np.sum(np.expm1(mean_y_train_ocean))

cv_y_train_land = std_y_train_land / mean_y_train_land
cv_y_train_ocean = std_y_train_ocean / mean_y_train_ocean

mean_integrated_ocean = np.nanmean(N_ing_train_ocean)
mean_integrated_land  = np.nanmean(N_ing_train_land)

### do seprate based on winter and summer hemsphere
# ---------------------------------------------------------
# Triangle latitude
# ---------------------------------------------------------

gridfil = xr.open_dataset(
    "/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc"
)

clat_tri = np.rad2deg(gridfil.clat.values)

NH_mask_ocean = clat_tri[ntri_ocean] > 15
SH_mask_ocean = clat_tri[ntri_ocean] < -15
TROPICS_mask_ocean = np.abs(clat_tri[ntri_ocean]) <= 15

y_train_ocean_Winter_Hemisphere      = np.concatenate([y_train_ocean.reshape(len(ntri_ocean), 217, 144)[NH_mask_ocean, :108, :], 
																																		y_train_ocean.reshape(len(ntri_ocean), 217, 144)[SH_mask_ocean, 109:, :]], axis=0)
y_train_ocean_Tropics = np.concatenate([y_train_ocean.reshape(len(ntri_ocean), 217, 144)[TROPICS_mask_ocean, :108, :],
																																		y_train_ocean.reshape(len(ntri_ocean), 217, 144)[TROPICS_mask_ocean, 109:, :]], axis=0)

y_train_ocean_Summer_Hemisphere      = np.concatenate([y_train_ocean.reshape(len(ntri_ocean), 217, 144)[SH_mask_ocean, :108, :], 
																																		y_train_ocean.reshape(len(ntri_ocean), 217, 144)[NH_mask_ocean, 109:, :]], axis=0)

# --- Key numbers for text ---
print("=" * 60)
print("STATS TO CONFIRM/CITE IN TEXT (OCEAN ONLY)")
print("=" * 60)

print(f"\n1. Max event integrated energy (ocean): {max_integrated_ocean:.4e}")
print(f"   Max event integrated energy (land):  {max_integrated_land:.4e}")
land_vs_ocean_max_pct = (max_integrated_land / max_integrated_ocean - 1) * 100
print(f"   Land vs Ocean (max event): {land_vs_ocean_max_pct:+.1f}%")

print(f"\n2. Mean spectrum integrated energy (ocean): {mean_integrated_ocean:.4e}")
print(f"   Mean spectrum integrated energy (land):  {mean_integrated_land:.4e}")
land_vs_ocean_mean_pct = (mean_integrated_land / mean_integrated_ocean - 1) * 100
print(f"   Land vs Ocean (mean spectrum): {land_vs_ocean_mean_pct:+.1f}%")

print(f"\n3. Ratio: max event / typical (mean) event, OCEAN:")
print(f"   {max_integrated_ocean / mean_integrated_ocean:.1f}x")
print(f"   Ratio: max event / typical (mean) event, LAND:")
print(f"   {max_integrated_land / mean_integrated_land:.1f}x")

# --- Locate peak bin in mean/std for ocean (center vs edge comparison) ---
print(f"\n4. OCEAN mean spectrum: max value and location")
flat_idx = np.unravel_index(np.argmax(mean_y_train_ocean), mean_y_train_ocean.shape)
print(f"   Peak mean bin at (k_idx,l_idx,m_idx)={flat_idx}, value={mean_y_train_ocean[flat_idx]:.3f}")

print(f"\n5. OCEAN std spectrum: max value and location")
flat_idx_std = np.unravel_index(np.argmax(std_y_train_ocean), std_y_train_ocean.shape)
print(f"   Peak std bin at (k_idx,l_idx,m_idx)={flat_idx_std}, value={std_y_train_ocean[flat_idx_std]:.3f}")

print(f"\n6. OCEAN: mean value AT the peak-std bin location (checking low-mean/high-var claim):")
print(f"   mean={mean_y_train_ocean[flat_idx_std]:.3f}, std={std_y_train_ocean[flat_idx_std]:.3f}, "
      f"CV={cv_y_train_ocean[flat_idx_std]:.3f}")

print(f"\n7. OCEAN: std value AT the peak-mean (center) bin location (checking high-mean/low-var claim):")
print(f"   mean={mean_y_train_ocean[flat_idx]:.3f}, std={std_y_train_ocean[flat_idx]:.3f}, "
      f"CV={cv_y_train_ocean[flat_idx]:.3f}")

for j, lam in enumerate(m):
    print(f"\nlambda_z index={j} (lam={lam}): OCEAN mean spectrum, center 2x2 block:\n"
          f"{mean_y_train_ocean[2:4, 2:4, j]}")
    # "How concentrated is the peak" - e.g., fraction of total energy in the single max bin vs. spread out
ocean_lz12 = max_y_train_ocean[:, :, 0]  # log-space, lambda_z=12km slice
print("Ocean lz=12km: max value =", ocean_lz12.max(), " mean value =", ocean_lz12.mean(), 
      " std =", ocean_lz12.std())
print("Max bin as fraction of total (log-space sum):", ocean_lz12.max() / ocean_lz12.sum())

# Compare activity level across lambda_z for this specific event (not the mean spectrum)
for j, lam in enumerate(m):
    slice_j = max_y_train_ocean[:, :, j]
    print(f"lambda_z index {j}: mean={slice_j.mean():.3f}, max={slice_j.max():.3f}, std={slice_j.std():.3f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np


def plot_spectral_bins(
    field_ocean,
    field_land,
    k,
    l,
    m,
    cbar_label,
    savename,
    cmap="RdBu_r",
    log_transform=False,
    vmin=None,
    vmax=None,
    symmetric=False,
    fontsize=11,
    n_ticks=3,
):
    field_ocean = np.nan_to_num(field_ocean, nan=0.0)
    field_land = np.nan_to_num(field_land, nan=0.0)

    if log_transform:
        field_ocean = np.log10(1 + field_ocean)
        field_land = np.log10(1 + field_land)

    if vmin is None or vmax is None:
        data_min = min(field_ocean.min(), field_land.min())
        data_max = max(field_ocean.max(), field_land.max())
        if vmin is None:
            vmin = data_min
        if vmax is None:
            vmax = data_max

    if symmetric:
        vabs = max(abs(vmin), abs(vmax))
        vmin, vmax = -vabs, vabs

    extent = [k.min(), k.max(), l.min(), l.max()]
    nm = len(m)

    fig, axes = plt.subplots(
        2,
        nm,
        figsize=(7.0, 3.8),
        sharex="col",
        sharey="row",
        constrained_layout=True,
    )

    if nm == 1:
        axes = axes.reshape(2, 1)

    for j in range(nm):
        lambda_km = 2 * np.pi / np.abs(m[j]) / 1000.0
        fields = [field_ocean[:, :, j], field_land[:, :, j]]

        for i in range(2):
            ax = axes[i, j]

            im = ax.imshow(
                fields[i].T,
                origin="lower",
                extent=extent,
                interpolation="nearest",
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
            )

            ax.axhline(0, color="k", lw=0.6, alpha=0.4)
            ax.axvline(0, color="k", lw=0.6, alpha=0.4)

            # Subpanel headers
            if i == 0:
                ax.set_title(
                    rf"$\lambda_z = {lambda_km:.1f}\text{{ km}}$",
                    fontsize=fontsize,
                    pad=6,
                )

            # Row identity labels (Ocean / Land)
            if j == 0:
                label_prefix = "Ocean" if i == 0 else "Land"
                ax.set_ylabel(label_prefix, fontsize=fontsize, labelpad=8)

            # Tick locators
            ax.xaxis.set_major_locator(mticker.MaxNLocator(n_ticks))
            ax.yaxis.set_major_locator(mticker.MaxNLocator(n_ticks))

            # Integer tick formatting (-1, 0, 1)
            ax.xaxis.set_major_formatter(
                mticker.FuncFormatter(lambda x, pos: f"{int(round(x*1e4))}")
            )
            ax.yaxis.set_major_formatter(
                mticker.FuncFormatter(lambda y, pos: f"{int(round(y*1e4))}")
            )

            ax.tick_params(labelsize=fontsize - 1)

    # Single centered figure-level axis labels
    fig.supxlabel(r"$k \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize)
    fig.supylabel(r"$\ell \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize)

    # Colorbar
    cbar = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
    cbar.set_label(cbar_label, fontsize=fontsize, fontweight="normal")
    cbar.ax.tick_params(labelsize=fontsize - 1)

    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.pdf",
        dpi=400,
        bbox_inches="tight",
    )
    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.png",
        dpi=400,
        bbox_inches="tight",
    )
    plt.show()

plot_spectral_bins(
    field_ocean=max_y_train_ocean,
    field_land=max_y_train_land,
    k=k,
    l=l,
    m=m,
    cbar_label=r"$\log\left(1 + \mathcal{N}_{\mathrm{max}} / \mathrm{kg\,m^{-1}\,s^{-1}}\right)$",
    savename="MaxSpectrum_OceanLand",
    cmap="viridis",
    log_transform=False,
)

In [ ]:
# Ocean
flat_idx_ocean = np.argmax(N_ing_train_ocean.reshape(len(ntri_ocean), 217))
row_ocean, col_ocean = np.unravel_index(flat_idx_ocean, (len(ntri_ocean), 217))

# Land
flat_idx_land = np.argmax(N_ing_train_land.reshape(len(ntri_land), 217))
row_land, col_land = np.unravel_index(flat_idx_land, (len(ntri_land), 217))

gridfil = xr.open_dataset(
        "/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc"
    )

clat_ocean, clon_ocean = np.rad2deg(gridfil["clat"].data[ntri_ocean[row_ocean]]), np.rad2deg(gridfil["clon"].data[ntri_ocean[row_ocean]])
clat_land, clon_land = np.rad2deg(gridfil["clat"].data[ntri_land[row_land]]), np.rad2deg(gridfil["clon"].data[ntri_land[row_land]])

print(f"\nocean: max integrated event at (lat, lon) = ({clat_ocean:.2f}, {clon_ocean:.2f})")
print(f"land: max integrated event at (lat, lon) = ({clat_land:.2f}, {clon_land:.2f})")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np


def plot_mean_std_single_domain(
    mean_field,  # array [nk, nl, nm]
    std_field,  # array [nk, nl, nm]
    k,
    l,
    m,
    domain_name,  # "Ocean" or "Land"
    savename,
    cmap="viridis",
    vlim_mean=None,  # (vmin, vmax) in log space
    vlim_std=None,  # (vmin, vmax) in log space
    use_percentile=True,
    fontsize=11,  # Standard paper size (9–11 pt)
    n_ticks=3,
):
    rows = ["Mean", "Std"]
    raw_fields = [mean_field, std_field]

    # Standard LaTeX math labels (no forced bolding)
    cbar_labels = [r"$\log\left(1+\mathcal{N}\,/\,\mathrm{kg\,m^{-1}\,s^{-1}}\right)$", r"$\sigma$"]

    # Log-transform once up front
    log_fields = list(raw_fields)

    def auto_vlim(arr):
        if use_percentile:
            return (np.nanpercentile(arr, 2), np.nanpercentile(arr, 98))
        return (np.nanmin(arr), np.nanmax(arr))

    vlims = [
        vlim_mean if vlim_mean is not None else auto_vlim(log_fields[0]),
        vlim_std if vlim_std is not None else auto_vlim(log_fields[1]),
    ]

    nm = len(m)
    n_rows = len(rows)

    # Standard publication width & compact height
    fig, axes = plt.subplots(
        n_rows,
        nm,
        figsize=(7.0, 3.8),
        sharex="col",
        sharey="row",
        constrained_layout=True,
    )

    if nm == 1:
        axes = axes.reshape(n_rows, 1)

    extent = [k.min(), k.max(), l.min(), l.max()]

    for row, (row_name, log_field, (vmin, vmax)) in enumerate(
        zip(rows, log_fields, vlims)
    ):
        im = None

        for j in range(nm):
            lambda_km = 2 * np.pi / np.abs(m[j]) / 1000.0
            ax = axes[row, j]

            im = ax.imshow(
                log_field[:, :, j].T,
                origin="lower",
                extent=extent,
                interpolation="nearest",
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
            )
            ax.axhline(0, color="k", lw=0.5, alpha=0.3)
            ax.axvline(0, color="k", lw=0.5, alpha=0.3)

            # Subpanel column headers (top row only)
            if row == 0:
                ax.set_title(
                    rf"$\lambda_z = {lambda_km:.1f}\text{{ km}}$",
                    fontsize=fontsize,
                    pad=6,
                )

            # Row identity label ("Mean" / "Std") on leftmost axis
            if j == 0:
                ax.set_ylabel(row_name, fontsize=fontsize, labelpad=8)

            # Tick locators
            ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=n_ticks))
            ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=n_ticks))

            # Integer tick formatting (-1, 0, 1) to match Option 2
            ax.xaxis.set_major_formatter(
                mticker.FuncFormatter(lambda x, pos: f"{int(round(x*1e4))}")
            )
            ax.yaxis.set_major_formatter(
                mticker.FuncFormatter(lambda y, pos: f"{int(round(y*1e4))}")
            )

            ax.tick_params(labelsize=fontsize - 1)

        # Row colorbars with regular weight labels
        cbar = fig.colorbar(im, ax=axes[row, :], fraction=0.02, pad=0.01)
        cbar.set_label(cbar_labels[row], fontsize=fontsize, fontweight="normal")
        cbar.ax.tick_params(labelsize=fontsize - 1)

    # Single centered figure-level axis labels
    fig.supxlabel(r"$k \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize)
    fig.supylabel(r"$\ell \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize)

    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.png",
        dpi=400,
        bbox_inches="tight",
    )
    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.pdf",
        dpi=400,
        bbox_inches="tight",
    )
    plt.show()
    
plot_mean_std_single_domain(
    mean_field=mean_y_train_ocean, std_field=std_y_train_ocean,
    k=k, l=l, m=m, domain_name="Ocean",
    cmap="viridis",
    vlim_mean=None, vlim_std=None,
    savename="ocean_mean_std_spectral",
)

plot_mean_std_single_domain(
    mean_field=mean_y_train_land, std_field=std_y_train_land,
    k=k, l=l, m=m, domain_name="Land",
    cmap="viridis",
    vlim_mean=None, vlim_std=None,
    savename="land_mean_std_spectral",
)

In [ ]:
# ============================================================
# CHECK 1: what exactly is plotted, and does it match the stats?
# ============================================================
for name, arr in [("mean_ocean", mean_y_train_ocean), ("std_ocean", std_y_train_ocean),
                  ("mean_land", mean_y_train_land),   ("std_land", std_y_train_land)]:
    print(f"{name}: shape={arr.shape} min={np.nanmin(arr):.3f} max={np.nanmax(arr):.3f}")

print("k:", k); print("l:", l); print("m:", m)
print("y_train_ocean shape:", y_train_ocean.shape,
      " global min/max:", np.nanmin(y_train_ocean), np.nanmax(y_train_ocean))

# ============================================================
# CHECK 2: candidates for the old "10.01" (and peak location of each)
# ============================================================
yo = y_train_ocean.reshape(-1, 6, 6, 4)

def peak(label, arr):
    idx = np.unravel_index(np.nanargmax(arr), arr.shape)
    print(f"{label:55s} max={np.nanmax(arr):8.3f} at {idx}")

peak("A) nanmean of log(1+N) per bin (what you plot now)", mean_y_train_ocean)
peak("B) log1p(mean of N) per bin (log of the true mean)",
     np.log1p(np.nanmean(np.expm1(yo), axis=0)))
peak("C) mean over samples of log1p(sum_k,l N) per lambda_z",
     np.nanmean(np.log1p(np.expm1(yo).sum(axis=(1, 2))), axis=0)[None, None, :])
peak("D) mean over (k,l) of A, per lambda_z",
     mean_y_train_ocean.mean(axis=(0, 1))[None, None, :])
peak("E) median over samples per bin", np.nanmedian(yo, axis=0))

# ============================================================
# CHECK 3: std/CV at the peak bins, for the text
# ============================================================
i_mean = np.unravel_index(np.nanargmax(mean_y_train_ocean), mean_y_train_ocean.shape)
i_std  = np.unravel_index(np.nanargmax(std_y_train_ocean),  std_y_train_ocean.shape)
for lab, i in [("peak mean", i_mean), ("peak std", i_std)]:
    mu, sd = mean_y_train_ocean[i], std_y_train_ocean[i]
    print(f"{lab}: idx={i} k={k[i[0]]:.3e} l={l[i[1]]:.3e} lz={m[i[2]]:.3e} "
          f"mean={mu:.3f} std={sd:.3f} CV={sd/mu:.3f}")

# ============================================================
# CHECK 4: are low edge means caused by filtered (zero) bins?
# ============================================================
zero_frac = (yo == 0).mean(axis=0)
print("\nOcean zero fraction, lambda_z=12 km (rows=k, cols=l):")
print(np.round(zero_frac[:, :, 0], 3))
print("Ocean zero fraction, lambda_z=7 km:")
print(np.round(zero_frac[:, :, 1], 3))
mean_nonzero = np.where(yo > 0, yo, np.nan)
print("Peak of mean over NON-zero samples only:",
      np.nanmax(np.nanmean(mean_nonzero, axis=0)))

# ============================================================
# CHECK 5: land vs ocean percentage, both definitions
# ============================================================
yl = y_train_land.reshape(-1, 6, 6, 4)
a = np.expm1(mean_y_train_land).sum() / np.expm1(mean_y_train_ocean).sum() - 1
b = np.nanmean(np.expm1(yl).sum(axis=(1,2,3))) / np.nanmean(np.expm1(yo).sum(axis=(1,2,3))) - 1
print(f"\nLand vs ocean, log-mean spectrum: {a*100:+.1f}%")
print(f"Land vs ocean, true mean of N:    {b*100:+.1f}%")

In [ ]:
### mean and std for winter, summer and tropics
y_train_ocean_Winter_Hemisphere_mean = np.mean(y_train_ocean_Winter_Hemisphere, axis=(0, 1)).reshape(6, 6, 4)
y_train_ocean_Winter_Hemisphere_std = np.std(y_train_ocean_Winter_Hemisphere, axis=(0, 1)).reshape(6, 6, 4)
y_train_ocean_Tropics_mean = np.mean(y_train_ocean_Tropics, axis=(0, 1)).reshape(6, 6, 4)
y_train_ocean_Tropics_std = np.std(y_train_ocean_Tropics, axis=(0, 1)).reshape(6, 6, 4)
y_train_ocean_Summer_Hemisphere_mean = np.mean(y_train_ocean_Summer_Hemisphere, axis=(0, 1)).reshape(6, 6, 4)
y_train_ocean_Summer_Hemisphere_std = np.std(y_train_ocean_Summer_Hemisphere, axis=(0, 1)).reshape(6, 6, 4)

import numpy as np

def integrated_energy(mean_field, k, l, m, mode="sum"):
    """
    mean_field: [nk, nl, nm]
    Returns an array of length nm: energy integrated over (k, l) for each m.
    mode="sum"  -> plain sum over the spectral grid
    mode="integral" -> multiplied by dk*dl (true spectral integral)
    """
    E = mean_field.sum(axis=(0, 1))
    if mode == "integral":
        dk = np.abs(np.diff(k)).mean()
        dl = np.abs(np.diff(l)).mean()
        E = E * dk * dl
    return E


regions = {
    "Winter":  y_train_ocean_Winter_Hemisphere_mean,
    "Tropics": y_train_ocean_Tropics_mean,
    "Summer":  y_train_ocean_Summer_Hemisphere_mean,
}

lambda_km = 2 * np.pi / np.abs(m) / 1000.0

energies = {name: integrated_energy(f, k, l, m) for name, f in regions.items()}

# ---- print table ----
header = f"{'lambda_z [km]':>14} | " + " | ".join(f"{name:>12}" for name in regions)
print(header)
print("-" * len(header))
for j in range(len(m)):
    row = f"{lambda_km[j]:>14.2f} | " + " | ".join(f"{energies[n][j]:>12.4e}" for n in regions)
    print(row)

print("-" * len(header))
print(f"{'Total':>14} | " + " | ".join(f"{energies[n].sum():>12.4e}" for n in regions))

# ---- relative share of each lambda_z within a region (%) ----
print("\nFraction of region total energy per vertical wavelength [%]")
print(f"{'lambda_z [km]':>14} | " + " | ".join(f"{name:>12}" for name in regions))
for j in range(len(m)):
    row = f"{lambda_km[j]:>14.2f} | " + " | ".join(
        f"{100 * energies[n][j] / energies[n].sum():>12.2f}" for n in regions
    )
    print(row)

# ---- ratios relative to Tropics ----
print("\nRatio to Tropics")
print(f"{'lambda_z [km]':>14} | {'Winter/Trop':>12} | {'Summer/Trop':>12}")
for j in range(len(m)):
    print(f"{lambda_km[j]:>14.2f} | "
          f"{energies['Winter'][j] / energies['Tropics'][j]:>12.3f} | "
          f"{energies['Summer'][j] / energies['Tropics'][j]:>12.3f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np


def plot_stat_by_region(
    fields,            # list of arrays [nk, nl, nm], one per region
    row_names,         # e.g. ["Winter", "Tropics", "Summer"]
    k,
    l,
    m,
    stat_name,         # "Mean" or "Std"
    cbar_label,        # e.g. r"$\log(1+\mathcal{N})$" or r"$\sigma$"
    savename,
    cmap="viridis",
    vlim=None,         # (vmin, vmax) in log space
    use_percentile=True,
    shared_scale=True, # same color scale across all rows
    fontsize=11,
    n_ticks=3,
):
    # Log-transform once up front
    log_fields = [np.log1p(f) for f in fields]

    def auto_vlim(arr):
        if use_percentile:
            return (np.nanpercentile(arr, 2), np.nanpercentile(arr, 98))
        return (np.nanmin(arr), np.nanmax(arr))

    # One shared scale across regions (so rows are directly comparable),
    # or one scale per row
    if vlim is not None:
        vlims = [vlim] * len(log_fields)
    elif shared_scale:
        vlims = [auto_vlim(np.stack(log_fields))] * len(log_fields)
    else:
        vlims = [auto_vlim(f) for f in log_fields]

    nm = len(m)
    n_rows = len(row_names)

    fig, axes = plt.subplots(
        n_rows,
        nm,
        figsize=(7.0, 5.4),
        sharex="col",
        sharey="row",
        constrained_layout=True,
    )

    if nm == 1:
        axes = axes.reshape(n_rows, 1)

    extent = [k.min(), k.max(), l.min(), l.max()]
    im = None

    for row, (row_name, log_field, (vmin, vmax)) in enumerate(
        zip(row_names, log_fields, vlims)
    ):
        for j in range(nm):
            lambda_km = 2 * np.pi / np.abs(m[j]) / 1000.0
            ax = axes[row, j]

            im = ax.imshow(
                log_field[:, :, j].T,
                origin="lower",
                extent=extent,
                interpolation="nearest",
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
            )
            ax.axhline(0, color="k", lw=0.5, alpha=0.3)
            ax.axvline(0, color="k", lw=0.5, alpha=0.3)

            # Column headers (top row only)
            if row == 0:
                ax.set_title(
                    rf"$\lambda_z = {lambda_km:.1f}\text{{ km}}$",
                    fontsize=fontsize,
                    pad=6,
                )

            # Region label on leftmost axis
            if j == 0:
                ax.set_ylabel(row_name, fontsize=fontsize, labelpad=8)

            ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=n_ticks))
            ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=n_ticks))

            ax.xaxis.set_major_formatter(
                mticker.FuncFormatter(lambda x, pos: f"{int(round(x*1e4))}")
            )
            ax.yaxis.set_major_formatter(
                mticker.FuncFormatter(lambda y, pos: f"{int(round(y*1e4))}")
            )
            ax.tick_params(labelsize=fontsize - 1)

        # Per-row colorbars only if scales differ between rows
        if not shared_scale and vlim is None:
            cbar = fig.colorbar(im, ax=axes[row, :], fraction=0.02, pad=0.01)
            cbar.set_label(cbar_label, fontsize=fontsize, fontweight="normal")
            cbar.ax.tick_params(labelsize=fontsize - 1)

    # Single colorbar for the whole figure when the scale is shared
    if shared_scale or vlim is not None:
        cbar = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.01)
        cbar.set_label(cbar_label, fontsize=fontsize, fontweight="normal")
        cbar.ax.tick_params(labelsize=fontsize - 1)

    fig.supxlabel(r"$k \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize)
    fig.supylabel(r"$\ell \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize)

    out = "/work/bb1097/b383521/plots/paper_plots"
    plt.savefig(f"{out}/{savename}.png", dpi=400, bbox_inches="tight")
    plt.savefig(f"{out}/{savename}.pdf", dpi=400, bbox_inches="tight")
    plt.show()


# ---- mean and std for winter, tropics and summer (ocean) ----
regions = ["Winter", "Tropics", "Summer"]

means = [
    y_train_ocean_Winter_Hemisphere_mean,
    y_train_ocean_Tropics_mean,
    y_train_ocean_Summer_Hemisphere_mean,
]
stds = [
    y_train_ocean_Winter_Hemisphere_std,
    y_train_ocean_Tropics_std,
    y_train_ocean_Summer_Hemisphere_std,
]

# Panel 1: mean
plot_stat_by_region(
    fields=means, row_names=regions, k=k, l=l, m=m,
    stat_name="Mean", cbar_label=r"$\log(1+\mathcal{N})$",
    savename="ocean_mean_spectral_regions",
)

# Panel 2: std
plot_stat_by_region(
    fields=stds, row_names=regions, k=k, l=l, m=m,
    stat_name="Std", cbar_label=r"$\sigma$",
    savename="ocean_std_spectral_regions",
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np


def plot_multi_metric_single_domain(
    fields_dict,
    k,
    l,
    m,
    domain_name,
    cmaps_dict,
    vlims_dict,
    savename,
    fontsize=11,  # Standard paper size (9–11 pt)
    n_ticks=3,
):
    n_metrics = len(fields_dict)
    nm = len(m)

    # Calculate dynamic height based on row count to maintain publication scaling
    fig_height = 1.9 * n_metrics

    fig, axes = plt.subplots(
        n_metrics,
        nm,
        figsize=(7.0, fig_height),
        sharex="col",
        sharey="row",
        constrained_layout=True,
    )

    if n_metrics == 1:
        axes = axes.reshape(1, nm)

    extent = [k.min(), k.max(), l.min(), l.max()]

    for row, (metric_name, field) in enumerate(fields_dict.items()):
        cmap = cmaps_dict[metric_name]
        vmin, vmax = vlims_dict[metric_name]
        im = None

        for j in range(nm):
            lambda_km = 2 * np.pi / np.abs(m[j]) / 1000.0
            ax = axes[row, j]

            im = ax.imshow(
                field[:, :, j].T,
                origin="lower",
                extent=extent,
                interpolation="nearest",
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
            )
            ax.axhline(0, color="k", lw=0.5, alpha=0.3)
            ax.axvline(0, color="k", lw=0.5, alpha=0.3)

            # Subpanel headers (top row only)
            if row == 0:
                ax.set_title(
                    rf"$\lambda_z = {lambda_km:.1f}\text{{ km}}$",
                    fontsize=fontsize,
                    pad=6,
                )

            # Row identity label (e.g. R2, MSE, CV) on leftmost axis
            if j == 0:
                ax.set_ylabel(metric_name, fontsize=fontsize, labelpad=8)

            # Tick locators
            ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=n_ticks))
            ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=n_ticks))

            # Scaled integer tick formatting (-1, 0, 1)
            ax.xaxis.set_major_formatter(
                mticker.FuncFormatter(lambda x, pos: f"{int(round(x*1e4))}")
            )
            ax.yaxis.set_major_formatter(
                mticker.FuncFormatter(lambda y, pos: f"{int(round(y*1e4))}")
            )

            ax.tick_params(labelsize=fontsize - 1)

        # Colorbar with unbolded metric label
        cbar = fig.colorbar(im, ax=axes[row, :], fraction=0.02, pad=0.01)
        cbar.set_label(metric_name, fontsize=fontsize, fontweight="normal")
        cbar.ax.tick_params(labelsize=fontsize - 1)

    # Single centered figure-level axis labels
    fig.supxlabel(r"$k \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize)
    fig.supylabel(r"$\ell \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize)

    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.png",
        dpi=400,
        bbox_inches="tight",
    )
    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.pdf",
        dpi=400,
        bbox_inches="tight",
    )
    plt.show()
## spectral bin
R2_score_ocean = metrics_data['ocean']["combined"]["R2_test"].reshape(6, 6, 4)
R2_score_land  = metrics_data['land']["combined"]["R2_test"].reshape(6, 6, 4)
MSE_score_ocean = metrics_data['ocean']["combined"]["mse_test"].reshape(6, 6, 4)
MSE_score_land  = metrics_data['land']["combined"]["mse_test"].reshape(6, 6, 4)
mse_vmax = np.nanpercentile(
    np.concatenate([MSE_score_ocean.ravel(), MSE_score_land.ravel()]), 95
)
cv_vmax = np.nanpercentile(
    np.concatenate([cv_y_train_ocean.ravel(), cv_y_train_land.ravel()]), 95
)

print("mse_vmax:", mse_vmax)
print("cv_vmax:", cv_vmax)
# Check your actual value ranges first
print("Ocean R2 range:", R2_score_ocean.min(), R2_score_ocean.max())
print("Land R2 range:", R2_score_land.min(), R2_score_land.max())
print("Ocean MSE range:", MSE_score_ocean.min(), MSE_score_ocean.max())
print("Land MSE range:", MSE_score_land.min(), MSE_score_land.max())
cmaps_dict = {"R2": "RdBu_r", "MSE": "magma", "CV": "magma"}
vlims_dict = {"R2": (0, 1), "MSE": (0, mse_vmax), "CV": (0, cv_vmax)}
# Usage:
# Ocean panel
with  xr.open_dataset(f"/work/bb1097/b383521/GW_Data/coarsed_action/coarse_gw_spectrum_group_{1}_fileslide_0.nc") as ds:
      k  = ds.k.data
      l  = ds.l.data
      m  = ds.m.data
# after re-running/restoring the original k, l, m wavenumber arrays:
k_wavenumber_array = k    # or whatever the correct array is, copy it to a safer name
l_wavenumber_array = l
m = np.asarray(m, dtype=float)
k = np.asarray(k, dtype=float)
l = np.asarray(l, dtype=float)

plot_multi_metric_single_domain(
    fields_dict={"R2": R2_score_ocean, "MSE": MSE_score_ocean, "CV": cv_y_train_ocean},
    k=k, l=l, m=m,
    domain_name="Ocean",
    cmaps_dict=cmaps_dict,
    vlims_dict=vlims_dict,
    savename="Ocean_Metric_Panel",
)

# Land panel
plot_multi_metric_single_domain(
    fields_dict={"R2": R2_score_land, "MSE": MSE_score_land, "CV": cv_y_train_land},
    k=k, l=l, m=m,
    domain_name="Land",
    cmaps_dict=cmaps_dict,
    vlims_dict=vlims_dict,
    savename="Land_Metric_Panel",
)

In [ ]:
print("Ocean R2 min:", R2_score_ocean.min(), "at index", np.unravel_index(np.argmin(R2_score_ocean), R2_score_ocean.shape))
print("Ocean R2 max:", R2_score_ocean.max(), "at index", np.unravel_index(np.argmax(R2_score_ocean), R2_score_ocean.shape))

# Average R2 per lambda_z (you asked if this is needed -- yes, it directly supports your "R2 decreases with shorter lambda_z" claim)
for j, lam in enumerate(m):
    print(f"lambda_z index {j}: mean R2 = {R2_score_ocean[:,:,j].mean():.3f}, "
          f"mean MSE = {MSE_score_ocean[:,:,j].mean():.3f}, mean CV = {cv_y_train_ocean[:,:,j].mean():.3f}")

from scipy.stats import spearmanr
import numpy as np

def compute_spearman_table(R2, MSE, CV, label=""):
    r2_flat = R2.ravel()
    mse_flat = MSE.ravel()
    cv_flat = CV.ravel()

    rho_mse, p_mse = spearmanr(mse_flat, r2_flat)
    rho_cv, p_cv = spearmanr(cv_flat, r2_flat)

    print(f"=== {label} ===")
    print(f"Spearman rho(MSE, R2) = {rho_mse:.3f}, p = {p_mse:.3e}")
    print(f"Spearman rho(CV, R2)  = {rho_cv:.3f}, p = {p_cv:.3e}")

    return {
        "rho_mse_r2": rho_mse, "p_mse_r2": p_mse,
        "rho_cv_r2": rho_cv, "p_cv_r2": p_cv,
    }

# --- Ocean ---
ocean_results = compute_spearman_table(
    R2_score_ocean, MSE_score_ocean, cv_y_train_ocean, label="OCEAN"
)

# --- Land ---
land_results = compute_spearman_table(
    R2_score_land, MSE_score_land, cv_y_train_land, label="LAND"
)

In [ ]:
import numpy as np

# Zonal (k) asymmetry: compare negative-k half vs positive-k half
# (excluding center column if grid has odd symmetry; adjust indices to your actual grid)
neg_k = slice(0, 3)   # k indices 0,1,2 = negative k
pos_k = slice(3, 6)   # k indices 3,4,5 = positive k

for name, arr in [("R2", R2_score_ocean), ("MSE", MSE_score_ocean), ("CV", cv_y_train_ocean)]:
    mean_neg_k = arr[neg_k, :, :].mean()
    mean_pos_k = arr[pos_k, :, :].mean()
    print(f"{name}: mean(negative k) = {mean_neg_k:.4f}, mean(positive k) = {mean_pos_k:.4f}, "
          f"diff = {mean_pos_k - mean_neg_k:+.4f}")

print()

# Meridional (l) symmetry: compare negative-l half vs positive-l half
neg_l = slice(0, 3)
pos_l = slice(3, 6)

for name, arr in [("R2", R2_score_ocean), ("MSE", MSE_score_ocean), ("CV", cv_y_train_ocean)]:
    mean_neg_l = arr[:, neg_l, :].mean()
    mean_pos_l = arr[:, pos_l, :].mean()
    print(f"{name}: mean(negative l) = {mean_neg_l:.4f}, mean(positive l) = {mean_pos_l:.4f}, "
          f"diff = {mean_pos_l - mean_neg_l:+.4f}")

In [ ]:
from scipy.stats import spearmanr
import numpy as np

# Flatten each 6x6x4 grid to compare land vs ocean directly, bin-by-bin
r2_ocean_flat = R2_score_ocean.ravel()
r2_land_flat = R2_score_land.ravel()
mse_ocean_flat = MSE_score_ocean.ravel()
mse_land_flat = MSE_score_land.ravel()
cv_ocean_flat = cv_y_train_ocean.ravel()
cv_land_flat = cv_y_train_land.ravel()

for name, a, b in [("R2", r2_ocean_flat, r2_land_flat),
                     ("MSE", mse_ocean_flat, mse_land_flat),
                     ("CV", cv_ocean_flat, cv_land_flat)]:
    rho, p = spearmanr(a, b)
    mean_abs_diff = np.mean(np.abs(a - b))
    print(f"{name}: Spearman(ocean,land) rho={rho:.3f}, p={p:.2e}, "
          f"mean|ocean-land|={mean_abs_diff:.4f}, "
          f"ocean_mean={a.mean():.3f}, land_mean={b.mean():.3f}")

In [ ]:
import pandas as pd
import numpy as np

def build_metrics_table(k, l, m, r2, mse, cv, mean, std):
    """
    Builds a long-format table with one row per (k, l, m) bin.
    r2, mse, cv, mean, std: arrays of shape (len(k), len(l), len(m))
    """
    rows = []
    for ik, kval in enumerate(k):
        for il, lval in enumerate(l):
            for im, mval in enumerate(m):
                rows.append({
                    "k": kval,
                    "l": lval,
                    "m": mval,
                    "Mean": mean[ik, il, im],
                    "Std": std[ik, il, im],
                    "R2": r2[ik, il, im],
                    "MSE": mse[ik, il, im],
                    "CV": cv[ik, il, im],
                })
    return pd.DataFrame(rows)


# --- Ocean table ---
df_ocean = build_metrics_table(
    k=k, l=l, m=m,
    r2=R2_score_ocean, mse=MSE_score_ocean, cv=cv_y_train_ocean,
    mean=mean_y_train_ocean, std=std_y_train_ocean,
)
print("=== OCEAN ===")
print(df_ocean.to_string(index=False))

# --- Land table ---
df_land = build_metrics_table(
    k=k, l=l, m=m,
    r2=R2_score_land, mse=MSE_score_land, cv=cv_y_train_land,
    mean=mean_y_train_land, std=std_y_train_land,
)
print("\n=== LAND ===")
print(df_land.to_string(index=False))

# Optional: save each to CSV
df_ocean.to_csv("/work/bb1097/b383521/plots/paper_plots/ocean_bin_metrics.csv", index=False)
df_land.to_csv("/work/bb1097/b383521/plots/paper_plots/land_bin_metrics.csv", index=False)

In [ ]:
from scipy.stats import spearmanr

rho, p = spearmanr(df_ocean["MSE"], df_ocean["R2"])
print(f"Ocean: Spearman rho(MSE, R2) = {rho:.3f}, p = {p:.4g}")

rho, p = spearmanr(df_land["MSE"], df_land["R2"])
print(f"Land: Spearman rho(MSE, R2) = {rho:.3f}, p = {p:.4g}")
rho_cv, p_cv = spearmanr(df_ocean["CV"], df_ocean["R2"])
print(f"Ocean: Spearman rho(CV, R2) = {rho_cv:.3f}, p = {p_cv:.4g}")

rho_cv, p_cv = spearmanr(df_land["CV"], df_land["R2"])
print(f"Land: Spearman rho(CV, R2) = {rho_cv:.3f}, p = {p_cv:.4g}")

In [ ]:
plot_mean_std_single_domain(
    mean_field=mean_y_train_ocean, std_field=std_y_train_ocean,
    k=k, l=l, m=m, domain_name="Ocean",
    cmap="viridis",
    vlim_mean=None, vlim_std=None,
    savename="ocean_mean_std_spectral",
)

plot_mean_std_single_domain(
    mean_field=mean_y_train_land, std_field=std_y_train_land,
    k=k, l=l, m=m, domain_name="Land",
    cmap="viridis",
    vlim_mean=None, vlim_std=None,
    savename="land_mean_std_spectral",
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_skill_bias_dual_axis(
    A_ocean, A_land, savename="ocean_land_skill_bias", fontsize=11
):
    """A_ocean, A_land: lists of dicts with keys 'regime', 'Skill', 'RelBias'"""
    # Standard 2-column paper width (7.0 in)
    fig, axes = plt.subplots(
        1, 2, figsize=(7.0, 3.2), sharey=True, constrained_layout=True
    )

    datasets = [("Ocean", A_ocean, axes[0]), ("Land", A_land, axes[1])]

    skill_color = "#1f77b4"
    bias_color = "#d62728"

    right_axes = []
    handles = []

    for i, (name, A, ax1) in enumerate(datasets):
        regimes = [d["regime"] for d in A]
        skill = [d["Skill"] for d in A]
        relbias = [d["RelBias"] for d in A]
        x = np.arange(len(regimes))

        ax2 = ax1.twinx()
        right_axes.append(ax2)

        ax1.axhline(0, color="gray", lw=0.8, ls=":", alpha=0.7)

        (l1,) = ax1.plot(
            x,
            skill,
            "o-",
            color=skill_color,
            lw=1.8,
            markersize=5,
            label="Skill",
        )
        (l2,) = ax2.plot(
            x,
            relbias,
            "s--",
            color=bias_color,
            lw=1.8,
            markersize=5,
            label="Relative Bias",
        )

        ax1.set_xticks(x)
        ax1.set_xticklabels(regimes, fontsize=fontsize - 1, rotation=0)

        # Unbolded subpanel headers
        ax1.set_title(name, fontsize=fontsize, pad=6)

        # Axis labels (unbolded)
        if i == 0:
            ax1.set_ylabel("Skill", color=skill_color, fontsize=fontsize)
            handles = [l1, l2]
        if i == 1:
            ax2.set_ylabel(
                "Relative Bias", color=bias_color, fontsize=fontsize
            )

        # Colored tick marks and regular font weights for labels
        ax1.tick_params(
            axis="y",
            colors=skill_color,
            labelcolor=skill_color,
            labelsize=fontsize - 1,
        )
        ax2.tick_params(
            axis="y",
            colors=bias_color,
            labelcolor=bias_color,
            labelsize=fontsize - 1,
        )

        ax1.grid(alpha=0.25, lw=0.5)
        ax1.spines["top"].set_visible(False)
        ax2.spines["top"].set_visible(False)

    # Match right-axis limits across both panels
    ylims = [ax.get_ylim() for ax in right_axes]
    ymin = min(l[0] for l in ylims)
    ymax = max(l[1] for l in ylims)
    for ax in right_axes:
        ax.set_ylim(ymin, ymax)

    # Hide right-axis tick labels on the left panel
    right_axes[0].tick_params(labelright=False)

    # Centered figure-level x-axis label (unbolded)
    fig.supxlabel("Energy percentile regime", fontsize=fontsize)

    # Clean upper legend
    fig.legend(
        handles=handles,
        labels=["Skill", "Relative Bias"],
        loc="lower center",
        ncol=2,
        bbox_to_anchor=(0.5, 1.01),
        fontsize=fontsize - 1,
        frameon=False,
    )

    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.png",
        dpi=400,
        bbox_inches="tight",
    )
    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.pdf",
        dpi=400,
        bbox_inches="tight",
    )
    plt.show()

# --- Usage ---
A_ocean = regimes["ocean"]["combined"]
A_land = regimes["land"]["combined"]

plot_skill_bias_dual_axis(A_ocean, A_land, savename="ocean_land_skill_bias")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Examine bias due to log transform
y_test_land_log = metrics_data["land"]["combined"]["y_test"]
y_pred_land_log = metrics_data["land"]["combined"]["y_pred_test"]
y_test_ocean_log = metrics_data["ocean"]["combined"]["y_test"]
y_pred_ocean_log = metrics_data["ocean"]["combined"]["y_pred_test"]

# Per-sample log-space residual (mean over the 144 spectral dims) -> shape (n_samples,)
log_resid_ocean = np.mean(y_pred_ocean_log - y_test_ocean_log, axis=1)
log_resid_land  = np.mean(y_pred_land_log  - y_test_land_log,  axis=1)

# Back-transform to real space (per-bin, still shape (n_samples, 144))
y_test_land_orig  = np.expm1(y_test_land_log)
y_pred_land_orig  = np.expm1(y_pred_land_log)
y_test_ocean_orig = np.expm1(y_test_ocean_log)
y_pred_ocean_orig = np.expm1(y_pred_ocean_log)

# Per-sample TOTAL energy (must match your regime table's energy definition)
E_test_ocean = y_test_ocean_orig.sum(axis=1)
E_test_land  = y_test_land_orig.sum(axis=1)

## define regime bounds + pull RelBias straight from the existing regimes dict
regimes_labels = ["P0-P50", "P50-P90", "P90-P99", "P99-P100"]
tags = ["land", "ocean"]
regime_bounds = {}
relbias_table = {}
for tag in tags:
    regime_bounds[tag] = []
    relbias_table[tag] = []
    for i, regime in enumerate(regimes_labels):
        E_min = regimes[tag]["combined"][i]["E_min"]
        E_max = regimes[tag]["combined"][i]["E_max"]
        regime_bounds[tag].append((regime, E_min, E_max))
        relbias_table[tag].append(regimes[tag]["combined"][i]["RelBias"])

# --- Compute log-space bias per regime (this is the new part) ---
logbias_table = {"ocean": [], "land": []}

for name, lo, hi in regime_bounds["ocean"]:
    mask = (E_test_ocean >= lo) & (E_test_ocean < hi)
    logbias_table["ocean"].append(np.mean(log_resid_ocean[mask]))

for name, lo, hi in regime_bounds["land"]:
    mask = (E_test_land >= lo) & (E_test_land < hi)
    logbias_table["land"].append(np.mean(log_resid_land[mask]))

print("Log-space bias (ocean):", logbias_table["ocean"])
print("Log-space bias (land):", logbias_table["land"])
print("Real-space RelBias (ocean):", relbias_table["ocean"])
print("Real-space RelBias (land):", relbias_table["land"])


import matplotlib.pyplot as plt
import numpy as np


def plot_logspace_vs_realspace_bias(
    regime_labels,
    logbias_ocean,
    relbias_ocean,
    logbias_land,
    relbias_land,
    savename="logspace_vs_realspace_bias",
    fontsize=11,  # Standard paper font size (9–11 pt)
):
    # Standard 2-column paper figure dimensions
    fig, axes = plt.subplots(
        1, 2, figsize=(7.0, 3.2), sharey=True, constrained_layout=True
    )

    datasets = [
        ("Ocean", logbias_ocean, relbias_ocean, axes[0]),
        ("Land", logbias_land, relbias_land, axes[1]),
    ]

    logbias_color = "#7b2d8e"
    relbias_color = "#d62728"
    x = np.arange(len(regime_labels))
    handles = []

    for i, (name, logbias, relbias, ax) in enumerate(datasets):
        ax.axhline(0, color="gray", lw=0.8, ls=":", alpha=0.7)

        (l1,) = ax.plot(
            x,
            logbias,
            "o-",
            color=logbias_color,
            lw=1.8,
            markersize=5,
            label="Log-space bias",
        )
        (l2,) = ax.plot(
            x,
            relbias,
            "s--",
            color=relbias_color,
            lw=1.8,
            markersize=5,
            label="Real-space RelBias",
        )

        ax.set_xticks(x)
        ax.set_xticklabels(regime_labels, fontsize=fontsize - 1, rotation=0)

        # Unbolded subpanel headers
        ax.set_title(name, fontsize=fontsize, pad=6)

        ax.tick_params(axis="both", labelsize=fontsize - 1)
        ax.grid(alpha=0.25, lw=0.5)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if i == 0:
            ax.set_ylabel("Bias", fontsize=fontsize)
            handles = [l1, l2]

    # Centered figure-level x-axis label (unbolded)
    fig.supxlabel("Energy percentile regime", fontsize=fontsize)

    # Clean upper legend
    fig.legend(
        handles=handles,
        labels=["Log-space bias", "Real-space RelBias"],
        loc="lower center",
        ncol=2,
        bbox_to_anchor=(0.5, 1.01),
        fontsize=fontsize - 1,
        frameon=False,
    )

    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.png",
        dpi=400,
        bbox_inches="tight",
    )
    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.pdf",
        dpi=400,
        bbox_inches="tight",
    )
    plt.show()


# --- Call statement ---
plot_logspace_vs_realspace_bias(
    regimes_labels,
    logbias_table["ocean"],
    relbias_table["ocean"],
    logbias_table["land"],
    relbias_table["land"],
    savename="logspace_vs_realspace_bias",
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_parity_side_by_side(
    y_true_ocean,
    y_pred_ocean,
    y_true_land,
    y_pred_land,
    savename="parity_ocean_land",
    fontsize=11,  # Standard paper font size (9–11 pt)
    bins=150,
):
    """y_true_*, y_pred_*: real-space arrays, any shape -- will be raveled."""
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(7.0, 3.2),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )

    datasets = [
        ("Ocean", y_true_ocean, y_pred_ocean, axes[0]),
        ("Land", y_true_land, y_pred_land, axes[1]),
    ]

    # Pre-calculate global min/max counts for a shared colorbar scale
    processed_data = []
    global_min_val = float("inf")
    global_max_val = float("-inf")

    for name, y_true_orig, y_pred_orig, ax in datasets:
        y_true_flat = y_true_orig.ravel()
        y_pred_flat = np.maximum(y_pred_orig.ravel(), 0)

        mask = (y_true_flat > 0) & (y_pred_flat > 0)
        yt = y_true_flat[mask]
        yp = y_pred_flat[mask]

        global_min_val = min(global_min_val, yt.min(), yp.min())
        global_max_val = max(global_max_val, yt.max(), yp.max())
        processed_data.append((name, yt, yp, ax))

    hb = None
    lims = [global_min_val, global_max_val]

    for name, yt, yp, ax in processed_data:
        hb = ax.hexbin(
            yt,
            yp,
            gridsize=bins,
            xscale="log",
            yscale="log",
            cmap="viridis",
            mincnt=1,
            bins="log",
        )

        ax.plot(lims, lims, "r--", lw=1.2, label="1:1 line")

        # Unbolded subpanel headers
        ax.set_title(name, fontsize=fontsize, pad=6)

        ax.tick_params(axis="both", labelsize=fontsize - 1)
        ax.grid(alpha=0.2, lw=0.5)

        if name == "Ocean":
            ax.legend(fontsize=fontsize - 1, frameon=False, loc="upper left")

    # Single centered figure-level axis labels
    fig.supxlabel(r"True $\mathcal{N}$", fontsize=fontsize)
    fig.supylabel(r"Predicted $\mathcal{N}$", fontsize=fontsize)

    # Shared figure-level colorbar
    cbar = fig.colorbar(hb, ax=axes, fraction=0.02, pad=0.01)
    cbar.set_label("Count", fontsize=fontsize, fontweight="normal")
    cbar.ax.tick_params(labelsize=fontsize - 1)

    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.png",
        dpi=400,
        bbox_inches="tight",
    )
    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.pdf",
        dpi=400,
        bbox_inches="tight",
    )
    plt.show()


# --- Call statement ---
plot_parity_side_by_side(
    y_test_ocean_orig,
    y_pred_ocean_orig,
    y_test_land_orig,
    y_pred_land_orig,
    savename="parity_ocean_land",
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MultipleLocator


def _prep_log10(y_true, y_pred):
    yt = y_true.ravel()
    yp = np.maximum(y_pred.ravel(), 0)
    m = (yt > 0) & (yp > 0)
    print(f"dropped {100 * (1 - m.mean()):.3f}% of points (true or pred <= 0)")
    return np.log10(yt[m]), np.log10(yp[m])


def _binned_percentiles(x, y, lo, hi, nb=36, min_n=50):
    edges = np.linspace(lo, hi, nb + 1)
    centers = 0.5 * (edges[1:] + edges[:-1])
    idx = np.digitize(x, edges) - 1
    out = np.full((3, nb), np.nan)
    for i in range(nb):
        v = y[idx == i]
        if v.size >= min_n:
            out[:, i] = np.percentile(v, [5, 50, 95])
    return centers, out


def plot_parity_side_by_side(
    y_true_ocean,
    y_pred_ocean,
    y_true_land,
    y_pred_land,
    savename="parity_ocean_land",
    fontsize=10,
    bins=110,
    lim=(-5, 8.5),          # log10 range of both axes (square panels)
    min_count=5,
    show_frac_1dex=False,   # optional third metric
):
    data = {
        "Ocean": _prep_log10(y_true_ocean, y_pred_ocean),
        "Land": _prep_log10(y_true_land, y_pred_land),
    }
    lo, hi = lim
    edges = np.linspace(lo, hi, bins + 1)

    for k, (t, p) in data.items():
        out = np.mean((t < lo) | (t > hi) | (p < lo) | (p > hi))
        print(f"{k}: {100 * out:.3f}% of points outside plotted range {lim}")

    H = {k: np.histogram2d(t, p, bins=[edges, edges])[0] for k, (t, p) in data.items()}
    norm = LogNorm(vmin=min_count, vmax=max(h.max() for h in H.values()))

    fig = plt.figure(figsize=(7.2, 6.4), constrained_layout=True)
    gs = fig.add_gridspec(
        2, 3, width_ratios=[1, 1, 0.05], height_ratios=[3, 1.2]
    )
    ax_top = [fig.add_subplot(gs[0, 0])]
    ax_top.append(fig.add_subplot(gs[0, 1], sharex=ax_top[0], sharey=ax_top[0]))
    ax_res = [fig.add_subplot(gs[1, 0], sharex=ax_top[0])]
    ax_res.append(fig.add_subplot(gs[1, 1], sharex=ax_top[0], sharey=ax_res[0]))
    cax = fig.add_subplot(gs[0, 2])

    pow10 = FuncFormatter(lambda v, _: rf"$10^{{{int(round(v))}}}$")
    unit = r"(kg m$^{-1}$ s$^{-1}$)"

    pm = None
    for col, name in enumerate(["Ocean", "Land"]):
        lt, lp = data[name]
        ax, axr = ax_top[col], ax_res[col]

        # ---- parity panel
        pm = ax.pcolormesh(
            edges, edges, np.ma.masked_less(H[name].T, min_count),
            norm=norm, cmap="viridis", rasterized=True,
        )
        ax.plot([lo, hi], [lo, hi], "r--", lw=1.1)

        c, q = _binned_percentiles(lt, lp, lo, hi)
        ax.fill_between(c, q[0], q[2], color="w", alpha=0.35, lw=0)
        ax.plot(c, q[1], color="w", lw=2.4)
        ax.plot(c, q[1], color="k", lw=1.2)

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_aspect("equal", adjustable="box")
        ax.set_title(name, fontsize=fontsize, pad=6)
        ax.grid(alpha=0.2, lw=0.5)
        ax.xaxis.set_major_locator(MultipleLocator(2))
        ax.yaxis.set_major_locator(MultipleLocator(2))
        ax.yaxis.set_major_formatter(pow10)
        ax.tick_params(labelsize=fontsize - 1, labelbottom=False)
        if col == 0:
            ax.set_ylabel(rf"Predicted $\mathcal{{N}}$ {unit}", fontsize=fontsize)
        else:
            ax.tick_params(labelleft=False)

        res = lp - lt
        r2 = 1 - np.mean(res**2) / np.var(lt)
        txt = rf"$R^2_{{\log}}$ = {r2:.2f}" "\n" f"MAE = {np.mean(np.abs(res)):.2f} dex"
        if show_frac_1dex:
            txt += f"\n|err| < 1 dex: {100 * np.mean(np.abs(res) < 1):.1f} %"
        ax.text(0.97, 0.04, txt, transform=ax.transAxes, ha="right", va="bottom",
                fontsize=fontsize - 1,
                bbox=dict(fc="w", ec="none", alpha=0.85, pad=2))

        # ---- residual panel: log10(pred) - log10(true)
        rc, rq = _binned_percentiles(lt, res, lo, hi)
        axr.axhline(0, color="r", ls="--", lw=1.0)
        axr.fill_between(rc, rq[0], rq[2], color="C0", alpha=0.25, lw=0)
        axr.plot(rc, rq[1], color="C0", lw=1.5)
        axr.set_ylim(-3, 5)
        axr.grid(alpha=0.2, lw=0.5)
        axr.xaxis.set_major_formatter(pow10)
        axr.tick_params(labelsize=fontsize - 1)
        axr.set_xlabel(rf"True $\mathcal{{N}}$ {unit}", fontsize=fontsize)
        if col == 0:
            axr.set_ylabel(r"Residual (dex)", fontsize=fontsize)
        else:
            axr.tick_params(labelleft=False)

    handles = [
        Line2D([], [], color="r", ls="--", lw=1.1, label="1:1 line"),
        Line2D([], [], color="k", lw=1.2, label="Median"),
        Patch(fc="0.6", alpha=0.5, label="5–95 % range"),
    ]
    ax_top[0].legend(handles=handles, fontsize=fontsize - 1, frameon=False,
                     loc="upper left")
    ax_res[0].legend(handles=[
        Line2D([], [], color="C0", lw=1.5, label="Median"),
        Patch(fc="C0", alpha=0.25, label="5–95 % range"),
    ], fontsize=fontsize - 2, frameon=False, loc="upper right")

    cb = fig.colorbar(pm, cax=cax)
    cb.set_label("Count", fontsize=fontsize)
    cb.ax.tick_params(labelsize=fontsize - 1)

    for ext in ("png", "pdf"):
        plt.savefig(f"/work/bb1097/b383521/plots/paper_plots/{savename}.{ext}",
                    dpi=400, bbox_inches="tight")
    plt.show()


plot_parity_side_by_side(
    y_test_ocean_orig, y_pred_ocean_orig,
    y_test_land_orig, y_pred_land_orig,
    savename="parity_ocean_land",
)

In [ ]:
import numpy as np

# --- Ocean ---
y_true_flat = y_test_ocean_orig.ravel()
y_pred_flat = np.maximum(y_pred_ocean_orig.ravel(), 0)

mask = (y_true_flat > 0) & (y_pred_flat > 0)
yt = y_true_flat[mask]
yp = y_pred_flat[mask]

threshold = np.percentile(yt, 90)   # 90th percentile of per-bin TRUE magnitude
low_mask = yt < threshold
high_mask = yt >= threshold

resid = yp - yt

print("=== OCEAN ===")
print(f"Threshold (P90 of true N): {threshold:.3f}")
print(f"Low-magnitude bins  (n={low_mask.sum()}): mean bias = {np.mean(resid[low_mask]):.4f}, "
      f"std = {np.std(resid[low_mask]):.4f}")
print(f"High-magnitude bins (n={high_mask.sum()}): mean bias = {np.mean(resid[high_mask]):.4f}, "
      f"std = {np.std(resid[high_mask]):.4f}")

# --- Land (repeat identically) ---
y_true_flat_l = y_test_land_orig.ravel()
y_pred_flat_l = np.maximum(y_pred_land_orig.ravel(), 0)

mask_l = (y_true_flat_l > 0) & (y_pred_flat_l > 0)
yt_l = y_true_flat_l[mask_l]
yp_l = y_pred_flat_l[mask_l]

threshold_l = np.percentile(yt_l, 90)
low_mask_l = yt_l < threshold_l
high_mask_l = yt_l >= threshold_l

resid_l = yp_l - yt_l

print("\n=== LAND ===")
print(f"Threshold (P90 of true N): {threshold_l:.3f}")
print(f"Low-magnitude bins  (n={low_mask_l.sum()}): mean bias = {np.mean(resid_l[low_mask_l]):.4f}, "
      f"std = {np.std(resid_l[low_mask_l]):.4f}")
print(f"High-magnitude bins (n={high_mask_l.sum()}): mean bias = {np.mean(resid_l[high_mask_l]):.4f}, "
      f"std = {np.std(resid_l[high_mask_l]):.4f}")

In [ ]:
import os
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.collections import PolyCollection


def plot_global_map_pair(
    var_top,
    var_bottom,
    save_dir,
    filename="global_flux_map_pair",
    title_top="True",
    title_bottom="Predicted",
    cbar_label=r"$\log(1+E_{GW})$",
    cmap="Reds",
    vmin=None,
    vmax=None,
    fontsize=11,  # Standard publication size
):
    gridfil = xr.open_dataset(
        "/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc"
    )

    clon_vertices = np.rad2deg(gridfil.clon_vertices.values)
    clat_vertices = np.rad2deg(gridfil.clat_vertices.values)

    clon_vertices = np.where(
        clon_vertices < -180.0, clon_vertices + 360.0, clon_vertices
    )
    clon_vertices = np.where(
        clon_vertices > 180.0, clon_vertices - 360.0, clon_vertices
    )
    triangles = np.stack([clon_vertices, clat_vertices], axis=-1)

    if vmin is None:
        vmin = min(np.nanmin(var_top), np.nanmin(var_bottom))
    if vmax is None:
        vmax = max(np.nanmax(var_top), np.nanmax(var_bottom))

    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    cmap_obj = plt.get_cmap(cmap)

    # 2 rows, 1 column for vertical layout (Standard paper width 7.0 in)
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(7.0, 5.8),
        subplot_kw=dict(projection=ccrs.PlateCarree()),
        constrained_layout=True,
    )

    datasets = [
        (var_top, title_top, axes[0]),
        (var_bottom, title_bottom, axes[1]),
    ]

    for var, subtitle, ax in datasets:
        colors = cmap_obj(norm(var))

        ax.set_global()
        ax.coastlines(linewidth=0.6, zorder=2)

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.4,
            color="dimgray",
            alpha=0.45,
            zorder=2,
        )
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {"size": fontsize - 2}
        gl.ylabel_style = {"size": fontsize - 2}

        coll = PolyCollection(
            triangles,
            array=None,
            fc=colors,
            edgecolors="none",
            linewidth=0.05,
            transform=ccrs.Geodetic(),
            zorder=0,
        )
        ax.add_collection(coll)

        # Unbolded subpanel title
        ax.set_title(subtitle, fontsize=fontsize, pad=4)

    # ScalarMappable for colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])

    # Vertical colorbar on the right with compact width and padding
    cbar = fig.colorbar(
        sm,
        ax=axes,
        orientation="vertical",
        fraction=0.03,
        pad=0.03,
        shrink=0.85,
    )
    cbar.set_label(cbar_label, fontsize=fontsize, fontweight="normal")
    cbar.ax.tick_params(labelsize=fontsize - 1)

    os.makedirs(save_dir, exist_ok=True)
    outpath_png = os.path.join(save_dir, f"{filename}.png")
    outpath_pdf = os.path.join(save_dir, f"{filename}.pdf")

    plt.savefig(outpath_png, dpi=400, bbox_inches="tight")
    plt.savefig(outpath_pdf, dpi=400, bbox_inches="tight")
    plt.show()
    print(f"--> saved to {outpath_png} and {outpath_pdf}")


# --- Usage: January ---
E_test_jan = np.mean(metrics_data["Energy_test"][:, :22], axis=1)
E_model_jan = np.mean(metrics_data["Energy_model"][:, :22], axis=1)

vmin_jan = min(np.min(np.log1p(E_test_jan)), np.min(np.log1p(E_model_jan)))
vmax_jan = max(np.max(np.log1p(E_test_jan)), np.max(np.log1p(E_model_jan)))

plot_global_map_pair(
    np.log1p(E_test_jan),
    np.log1p(E_model_jan),
    save_dir="/work/bb1097/b383521/plots/paper_plots/",
    filename="E_test_vs_model_jan",
    title_top="True (January)",
    title_bottom="Predicted (January)",
    cbar_label=r"$\log(1+E_{\mathrm{GW}})$",
    cmap="Reds",
    vmin=vmin_jan,
    vmax=vmax_jan,
)

# --- Usage: August ---
E_test_aug = np.mean(metrics_data["Energy_test"][:, 22:], axis=1)
E_model_aug = np.mean(metrics_data["Energy_model"][:, 22:], axis=1)

vmin_aug = min(np.min(np.log1p(E_test_aug)), np.min(np.log1p(E_model_aug)))
vmax_aug = max(np.max(np.log1p(E_test_aug)), np.max(np.log1p(E_model_aug)))


plot_global_map_pair(
    np.log1p(E_test_aug),
    np.log1p(E_model_aug),
    save_dir="/work/bb1097/b383521/plots/paper_plots/",
    filename="E_test_vs_model_aug",
    title_top="True (August)",
    title_bottom="Predicted (August)",
    cbar_label=r"$\log(1+E_{\mathrm{GW}})$",
    cmap="Reds",
    vmin=vmin_aug,
    vmax=vmax_aug,
)

# ----- print corrlation and bias for Jan and Aug
r_jan = np.corrcoef(E_test_jan, E_model_jan)[0, 1]
r_aug = np.corrcoef(E_test_aug, E_model_aug)[0, 1]
bias_jan = np.mean(E_model_jan - E_test_jan)/np.mean(E_test_jan)
bias_aug = np.mean(E_model_aug - E_test_aug)/np.mean(E_test_aug)
print(f"January: Spearman r = {r_jan:.3f}, Relative Bias	= {bias_jan:.3%}")
print(f"August: Spearman r = {r_aug:.3f}, Relative Bias = {bias_aug:.3%}")

In [ ]:
from matplotlib.colors import TwoSlopeNorm

def plot_global_map_triple(E_true, E_pred, save_dir, filename, season,
                           cmap="magma_r", ratio_lim=0.5, fontsize=11):
    gridfil = xr.open_dataset("/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc")
    lon = np.rad2deg(gridfil.clon_vertices.values)
    lat = np.rad2deg(gridfil.clat_vertices.values)
    lon = np.where(lon < -180, lon + 360, lon)
    lon = np.where(lon > 180, lon - 360, lon)
    tris = np.stack([lon, lat], axis=-1)

    lt, lp = np.log10(E_true), np.log10(E_pred)
    ratio = lp - lt                                    # dex, >0 = overestimate
    vmin, vmax = np.nanpercentile(np.concatenate([lt, lp]), [1, 99])

    panels = [
        (lt, f"True ({season})", plt.Normalize(vmin, vmax), cmap),
        (lp, f"Predicted ({season})", plt.Normalize(vmin, vmax), cmap),
        (ratio, "Predicted / True", TwoSlopeNorm(0, -ratio_lim, ratio_lim), "RdBu_r"),
    ]

    fig, axes = plt.subplots(3, 1, figsize=(7.0, 8.4),
                             subplot_kw=dict(projection=ccrs.PlateCarree()),
                             constrained_layout=True)

    for ax, (var, title, norm, cm) in zip(axes, panels):
        cmo = plt.get_cmap(cm)
        ax.set_global()
        ax.add_collection(PolyCollection(tris, fc=cmo(norm(var)), edgecolors="none",
                                         transform=ccrs.Geodetic(), zorder=0))
        ax.coastlines(linewidth=0.6, zorder=2)
        gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="dimgray", alpha=0.45)
        gl.top_labels = gl.right_labels = False
        gl.xlabel_style = gl.ylabel_style = {"size": fontsize - 2}
        ax.set_title(title, fontsize=fontsize, pad=4)
        sm = plt.cm.ScalarMappable(cmap=cmo, norm=norm); sm.set_array([])
        cb = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02, shrink=0.9,
                          extend="both")
        cb.ax.tick_params(labelsize=fontsize - 1)
        if cm == "RdBu_r":
            cb.set_label(r"$\log_{10}(E_\mathrm{pred}/E_\mathrm{true})$", fontsize=fontsize)
        else:
            cb.set_label(r"$\log_{10}E$ (E in kg m$^{-1}$ s$^{-2}$)", fontsize=fontsize)

    for ext in ("png", "pdf"):
        plt.savefig(os.path.join(save_dir, f"{filename}.{ext}"), dpi=400, bbox_inches="tight")
    plt.show()

plot_global_map_triple(E_test_jan, E_model_jan,
    "/work/bb1097/b383521/plots/paper_plots/", "E_test_vs_model_jan", "January")

# statistics to report
for n, (t, p) in {"Jan": (E_test_jan, E_model_jan), "Aug": (E_test_aug, E_model_aug)}.items():
    from scipy.stats import spearmanr
    print(n,
          "r_lin", np.corrcoef(t, p)[0, 1],
          "r_log", np.corrcoef(np.log10(t), np.log10(p))[0, 1],
          "spearman", spearmanr(t, p)[0],
          "bias_lin", (p.mean() - t.mean()) / t.mean(),
          "bias_log10", np.mean(np.log10(p / t)))

In [ ]:
import os
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection


def plot_global_map_jan_aug(
    E_true_jan, E_pred_jan, E_true_aug, E_pred_aug,
    save_dir,
    filename="E_test_vs_model_jan_aug",
    cmap="magma_r",
    pct=(1, 99),          # robust colour limits per month
    fontsize=10,
):
    gridfil = xr.open_dataset("/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc")
    lon = np.rad2deg(gridfil.clon_vertices.values)
    lat = np.rad2deg(gridfil.clat_vertices.values)
    lon = np.where(lon < -180, lon + 360, lon)
    lon = np.where(lon > 180, lon - 360, lon)
    tris = np.stack([lon, lat], axis=-1)

    months = {
        "January": (np.log10(E_true_jan), np.log10(E_pred_jan)),
        "August": (np.log10(E_true_aug), np.log10(E_pred_aug)),
    }

    fig, axes = plt.subplots(
        2, 2, figsize=(7.2, 4.8),
        subplot_kw=dict(projection=ccrs.PlateCarree()),
        constrained_layout=True,
    )
    cmo = plt.get_cmap(cmap)

    for col, (month, (lt, lp)) in enumerate(months.items()):
        vmin, vmax = np.nanpercentile(np.concatenate([lt, lp]), pct)
        norm = plt.Normalize(vmin, vmax)

        for row, (var, label) in enumerate([(lt, "True"), (lp, "Predicted")]):
            ax = axes[row, col]
            ax.set_global()
            ax.add_collection(PolyCollection(
                tris, fc=cmo(norm(var)), edgecolors="none",
                transform=ccrs.Geodetic(), zorder=0))
            ax.coastlines(linewidth=0.5, zorder=2)

            gl = ax.gridlines(draw_labels=True, linewidth=0.3,
                              color="dimgray", alpha=0.45, zorder=2)
            gl.top_labels = False
            gl.right_labels = False
            gl.left_labels = (col == 0)
            gl.bottom_labels = (row == 1)
            gl.xlabel_style = gl.ylabel_style = {"size": fontsize - 3}
            ax.set_title(f"{label} ({month})", fontsize=fontsize, pad=3)

        sm = plt.cm.ScalarMappable(cmap=cmo, norm=norm)
        sm.set_array([])
        cb = fig.colorbar(sm, ax=axes[:, col], orientation="horizontal",
                          fraction=0.06, pad=0.04, shrink=0.9, extend="both")
        cb.set_label(r"$\log_{10}E$ ($E$ in kg m$^{-1}$ s$^{-2}$)", fontsize=fontsize)
        cb.ax.tick_params(labelsize=fontsize - 1)

    os.makedirs(save_dir, exist_ok=True)
    for ext in ("png", "pdf"):
        plt.savefig(os.path.join(save_dir, f"{filename}.{ext}"),
                    dpi=400, bbox_inches="tight")
    plt.show()


plot_global_map_jan_aug(
    E_test_jan, E_model_jan, E_test_aug, E_model_aug,
    save_dir="/work/bb1097/b383521/plots/paper_plots/",
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_energy_scatter_seasons(
    E_true_summer,
    E_pred_summer,
    E_true_winter,
    E_pred_winter,
    E_true_tropics,
    E_pred_tropics,
    savename="hexbin_energy_seasons",
    gridsize=80,
    fontsize=11,
):

    datasets = [
        ("Summer", E_true_summer, E_pred_summer, 0),
        ("Winter", E_true_winter, E_pred_winter, 1),
        ("Tropics", E_true_tropics, E_pred_tropics, 2),
    ]

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(10.0, 3.2),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )

    # ---------------------------------------------------------
    # Process data and determine shared global axis limits
    # ---------------------------------------------------------
    processed_data = []

    global_min = float("inf")
    global_max = float("-inf")

    for title, E_true_full, E_pred_full, idx in datasets:

        E_true = E_true_full.ravel()
        E_pred = E_pred_full.ravel()

        mask = (
            np.isfinite(E_true)
            & np.isfinite(E_pred)
            & (E_true > 0)
            & (E_pred > 0)
        )

        x = E_true[mask]
        y = E_pred[mask]

        r = np.corrcoef(x, y)[0, 1]
        bias = np.mean(y) / np.mean(x) - 1

        global_min = min(global_min, x.min(), y.min())
        global_max = max(global_max, x.max(), y.max())

        processed_data.append(
            (title, x, y, r, bias, axes[idx])
        )

    lims = [global_min, global_max]

    hb = None

    # ---------------------------------------------------------
    # Plot
    # ---------------------------------------------------------
    for title, x, y, r, bias, ax in processed_data:

        hb = ax.hexbin(
            x,
            y,
            gridsize=gridsize,
            bins="log",
            mincnt=1,
            xscale="log",
            yscale="log",
            cmap="viridis",
        )

        # 1:1 line
        ax.plot(
            lims,
            lims,
            "r--",
            lw=1.2,
            label="1:1 line",
        )

        # Panel title
        ax.set_title(
            title,
            fontsize=fontsize,
            pad=6,
        )

        # Statistics
        ax.text(
            0.05,
            0.95,
            f"$r={r:.3f}$\nBias={bias:.3f}",
            transform=ax.transAxes,
            fontsize=fontsize - 2,
            va="top",
            bbox=dict(
                facecolor="white",
                edgecolor="gray",
                alpha=0.8,
                boxstyle="round,pad=0.3",
            ),
        )

        ax.tick_params(
            axis="both",
            labelsize=fontsize - 1,
        )

        ax.grid(
            alpha=0.2,
            lw=0.5,
        )

    # Legend only once
    axes[0].legend(
        fontsize=fontsize - 2,
        frameon=False,
        loc="lower right",
    )

    # ---------------------------------------------------------
    # Axis labels
    # ---------------------------------------------------------
    fig.supxlabel(
        r"$E_{\mathrm{truth}}$",
        fontsize=fontsize,
    )

    fig.supylabel(
        r"$E_{\mathrm{pred}}$",
        fontsize=fontsize,
    )

    # ---------------------------------------------------------
    # Shared colorbar
    # ---------------------------------------------------------
    cbar = fig.colorbar(
        hb,
        ax=axes,
        fraction=0.02,
        pad=0.01,
    )

    cbar.set_label(
        "Count",
        fontsize=fontsize,
        fontweight="normal",
    )

    cbar.ax.tick_params(
        labelsize=fontsize - 1,
    )

    # ---------------------------------------------------------
    # Save
    # ---------------------------------------------------------
    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.png",
        dpi=400,
        bbox_inches="tight",
    )

    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.pdf",
        dpi=400,
        bbox_inches="tight",
    )

    plt.show()

# ---------------------------------------------------------
# Energy data
# ---------------------------------------------------------

E_test_jan_full = metrics_data["Energy_test"][:, :22]
E_model_jan_full = metrics_data["Energy_model"][:, :22]

E_test_aug_full = metrics_data["Energy_test"][:, 22:]
E_model_aug_full = metrics_data["Energy_model"][:, 22:]


# ---------------------------------------------------------
# Triangle latitude
# ---------------------------------------------------------

gridfil = xr.open_dataset(
    "/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc"
)

clat_tri = np.rad2deg(gridfil.clat.values)


# ---------------------------------------------------------
# Latitude masks
# ---------------------------------------------------------

NH_mask = clat_tri > 15
SH_mask = clat_tri < -15
TROPICS_mask = np.abs(clat_tri) <= 15


# ---------------------------------------------------------
# SUMMER
# January = Southern Hemisphere
# August = Northern Hemisphere
# ---------------------------------------------------------

summer_Energy_test = np.concatenate(
    [
        E_test_jan_full[SH_mask, :],
        E_test_aug_full[NH_mask, :]
    ],
    axis=0
)

summer_Energy_model = np.concatenate(
    [
        E_model_jan_full[SH_mask, :],
        E_model_aug_full[NH_mask, :]
    ],
    axis=0
)


# ---------------------------------------------------------
# WINTER
# January = Northern Hemisphere
# August = Southern Hemisphere
# ---------------------------------------------------------

winter_Energy_test = np.concatenate(
    [
        E_test_jan_full[NH_mask, :],
        E_test_aug_full[SH_mask, :]
    ],
    axis=0
)

winter_Energy_model = np.concatenate(
    [
        E_model_jan_full[NH_mask, :],
        E_model_aug_full[SH_mask, :]
    ],
    axis=0
)


# ---------------------------------------------------------
# TROPICS
# January + August
# ---------------------------------------------------------

tropics_Energy_test = np.concatenate(
    [
        E_test_jan_full[TROPICS_mask, :],
        E_test_aug_full[TROPICS_mask, :]
    ],
    axis=0
)

tropics_Energy_model = np.concatenate(
    [
        E_model_jan_full[TROPICS_mask, :],
        E_model_aug_full[TROPICS_mask, :]
    ],
    axis=0
)


# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

plot_energy_scatter_seasons(
    summer_Energy_test,
    summer_Energy_model,
    winter_Energy_test,
    winter_Energy_model,
    tropics_Energy_test,
    tropics_Energy_model,
    savename="hexbin_energy_seasons",
)

#---------------------------------------------------------
# print mean energy for each season and region
mean_summer = np.mean(summer_Energy_test)
mean_winter = np.mean(winter_Energy_test)
mean_tropics = np.mean(tropics_Energy_test)
print(f"Mean energy (Summer): {mean_summer:.4f}")
print(f"Mean energy (Winter): {mean_winter:.4f}")
print(f"Mean energy (Tropics): {mean_tropics:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_energy_scatter_seasons(
    E_true_summer, E_pred_summer,
    E_true_winter, E_pred_winter,
    E_true_tropics, E_pred_tropics,
    savename="hexbin_energy_seasons",
    gridsize=60,
    fontsize=11,
    time_average=True,   # average over the time axis (axis=1) before pooling
):
    """Inputs have shape (n_triangles, n_time)."""
    unit = r"(kg m$^{-1}$ s$^{-2}$)"
    datasets = [
        ("Summer", E_true_summer, E_pred_summer),
        ("Winter", E_true_winter, E_pred_winter),
        ("Tropics", E_true_tropics, E_pred_tropics),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(10.0, 3.2), sharex=True, sharey=True,
                             constrained_layout=True)

    processed, gmin, gmax = [], np.inf, -np.inf
    for title, Et, Ep in datasets:
        if time_average:
            Et, Ep = Et.mean(axis=1), Ep.mean(axis=1)
        Et, Ep = Et.ravel(), Ep.ravel()

        m = np.isfinite(Et) & np.isfinite(Ep) & (Et > 0) & (Ep > 0)
        x, y = Et[m], Ep[m]

        r = np.corrcoef(x, y)[0, 1]
        bias = np.mean(y) / np.mean(x) - 1
        print(f"{title:8s} n={x.size:8d}  r={r:.3f}  rel. bias={100 * bias:.1f}%")

        gmin, gmax = min(gmin, x.min(), y.min()), max(gmax, x.max(), y.max())
        processed.append((title, x, y, r, bias))

    lims = [gmin, gmax]
    hb = None
    for ax, (title, x, y, r, bias) in zip(axes, processed):
        hb = ax.hexbin(x, y, gridsize=35, bins="log", mincnt=1,
                       xscale="log", yscale="log", cmap="viridis")
        ax.plot(lims, lims, "r--", lw=1.2, label="1:1 line")
        ax.set_title(title, fontsize=fontsize, pad=6)
        ax.text(0.05, 0.95, f"$r={r:.2f}$\nRel. bias $={100 * bias:.1f}\\,\\%$",
                transform=ax.transAxes, fontsize=fontsize - 2, va="top",
                bbox=dict(facecolor="white", edgecolor="gray", alpha=0.8,
                          boxstyle="round,pad=0.3"))
        ax.tick_params(labelsize=fontsize - 1)
        ax.grid(alpha=0.2, lw=0.5)

    axes[0].legend(fontsize=fontsize - 2, frameon=False, loc="lower right")

    label = "time-averaged " if time_average else ""
    fig.supxlabel(rf"True {label}$E_{{\mathrm{{GW}}}}$ {unit}", fontsize=fontsize)
    fig.supylabel(rf"Predicted {label}$E_{{\mathrm{{GW}}}}$ {unit}", fontsize=fontsize)

    cbar = fig.colorbar(hb, ax=axes, fraction=0.02, pad=0.01)
    cbar.set_label("Count", fontsize=fontsize)
    cbar.ax.tick_params(labelsize=fontsize - 1)

    for ext in ("png", "pdf"):
        plt.savefig(f"/work/bb1097/b383521/plots/paper_plots/{savename}.{ext}",
                    dpi=400, bbox_inches="tight")
    plt.show()


# data preparation (unchanged) ... then:
plot_energy_scatter_seasons(
    summer_Energy_test, summer_Energy_model,
    winter_Energy_test, winter_Energy_model,
    tropics_Energy_test, tropics_Energy_model,
    savename="hexbin_energy_seasons",
    time_average=True,
)

# for the sentence on the harder, time-resolved test:
plot_energy_scatter_seasons(
    summer_Energy_test, summer_Energy_model,
    winter_Energy_test, winter_Energy_model,
    tropics_Energy_test, tropics_Energy_model,
    savename="hexbin_energy_seasons_timeresolved",
    time_average=False,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_energy_scatter_seasons(
    E_true_summer, E_pred_summer,
    E_true_winter, E_pred_winter,
    E_true_tropics, E_pred_tropics,
    savename="scatter_energy_seasons",
    fontsize=11,
    time_average=True,   # average over the time axis (axis=1) before pooling
):
    """Inputs have shape (n_triangles, n_time)."""
    unit = r"(kg m$^{-1}$ s$^{-2}$)"
    datasets = [
        ("Summer", E_true_summer, E_pred_summer),
        ("Winter", E_true_winter, E_pred_winter),
        ("Tropics", E_true_tropics, E_pred_tropics),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(10.0, 3.4), sharex=True, sharey=True,
                             constrained_layout=True)

    processed, gmin, gmax = [], np.inf, -np.inf
    for title, Et, Ep in datasets:
        if time_average:
            Et, Ep = Et.mean(axis=1), Ep.mean(axis=1)
        Et, Ep = Et.ravel(), Ep.ravel()

        m = np.isfinite(Et) & np.isfinite(Ep) & (Et > 0) & (Ep > 0)
        x, y = Et[m], Ep[m]

        r = np.corrcoef(x, y)[0, 1]                 # linear space
        bias = np.mean(y) / np.mean(x) - 1          # linear space
        print(f"{title:8s} n={x.size:7d}  r={r:.3f}  rel. bias={100 * bias:.1f}%")

        gmin, gmax = min(gmin, x.min(), y.min()), max(gmax, x.max(), y.max())
        processed.append((title, x, y, r, bias))

    lims = [gmin * 0.8, gmax * 1.25]

    for ax, (title, x, y, r, bias) in zip(axes, processed):
        ax.scatter(x, y, s=3, alpha=0.25, c="C0", linewidths=0, rasterized=True)
        ax.plot(lims, lims, "r--", lw=1.2, label="1:1 line")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_aspect("equal", adjustable="box")
        ax.set_title(title, fontsize=fontsize, pad=6)
        ax.text(0.05, 0.95, f"$r={r:.2f}$\nRel. bias $={100 * bias:.1f}\\,\\%$",
                transform=ax.transAxes, fontsize=fontsize - 2, va="top",
                bbox=dict(facecolor="white", edgecolor="gray", alpha=0.85,
                          boxstyle="round,pad=0.3"))
        ax.tick_params(labelsize=fontsize - 1)
        ax.grid(alpha=0.2, lw=0.5)

    axes[0].legend(fontsize=fontsize - 2, frameon=False, loc="lower right")

    label = "time-averaged " if time_average else ""
    fig.supxlabel(rf"True {label}$E_{{\mathrm{{GW}}}}$ {unit}", fontsize=fontsize)
    fig.supylabel(rf"Predicted {label}$E_{{\mathrm{{GW}}}}$ {unit}", fontsize=fontsize)

    for ext in ("png", "pdf"):
        plt.savefig(f"/work/bb1097/b383521/plots/paper_plots/{savename}.{ext}",
                    dpi=400, bbox_inches="tight")
    plt.show()


# data preparation (unchanged) ... then:
plot_energy_scatter_seasons(
    summer_Energy_test, summer_Energy_model,
    winter_Energy_test, winter_Energy_model,
    tropics_Energy_test, tropics_Energy_model,
    savename="scatter_energy_seasons",
    time_average=True,
)

In [ ]:
import numpy as np

# Time-averaged (as used for the spatial maps) — real space
E_test_jan_mean = np.mean(metrics_data["Energy_test"][:, :22], axis=1)
E_test_aug_mean = np.mean(metrics_data["Energy_test"][:, 22:], axis=1)

print("=== Time-averaged energy (spatial map basis) ===")
print(f"January: mean E = {E_test_jan_mean.mean():.4e}, max = {E_test_jan_mean.max():.4e}")
print(f"August:  mean E = {E_test_aug_mean.mean():.4e}, max = {E_test_aug_mean.max():.4e}")
print(f"Ratio (Jan/Aug), mean: {E_test_jan_mean.mean() / E_test_aug_mean.mean():.2f}x")
print(f"Ratio (Jan/Aug), max:  {E_test_jan_mean.max() / E_test_aug_mean.max():.2f}x")

# Log-space equivalent (matches what's shown in the colorbar of your spatial maps)
log_E_test_jan = np.log1p(E_test_jan_mean)
log_E_test_aug = np.log1p(E_test_aug_mean)

print("\n=== log(1+E) space (matches map colorbar) ===")
print(f"January: mean log(1+E) = {log_E_test_jan.mean():.3f}, max = {log_E_test_jan.max():.3f}")
print(f"August:  mean log(1+E) = {log_E_test_aug.mean():.3f}, max = {log_E_test_aug.max():.3f}")

# --- Also check using the RAVELED (not time-averaged) energy, matching the hexbin scatter basis ---
E_test_jan_full = metrics_data["Energy_test"][:, :22].ravel()
E_test_aug_full = metrics_data["Energy_test"][:, 22:].ravel()

print("\n=== Full (raveled, not time-averaged) energy, matches hexbin scatter basis ===")
print(f"January: mean E = {E_test_jan_full.mean():.4e}")
print(f"August:  mean E = {E_test_aug_full.mean():.4e}")
print(f"Ratio (Jan/Aug), mean: {E_test_jan_full.mean() / E_test_aug_full.mean():.2f}x")

In [ ]:
import os
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.collections import PolyCollection
from matplotlib.colors import TwoSlopeNorm


def plot_global_map_flux_pair(
    var_top,
    var_bottom,
    save_dir,
    filename="global_momentum_flux_map_pair",
    title_top="True",
    title_bottom="Predicted",
    cbar_label=r"$\rho\overline{u'w'}$ [mPa]",
    cmap="RdBu_r",
    vmin=None,
    vmax=None,
    percentile_clip=True,
    p_low=2,
    p_high=98,
    symmetric=True,
    fontsize=11,
):
    var_top = np.asarray(var_top, dtype=float)
    var_bottom = np.asarray(var_bottom, dtype=float)

    gridfil = xr.open_dataset(
        "/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc"
    )
    clon_vertices = np.rad2deg(gridfil.clon_vertices.values)
    clat_vertices = np.rad2deg(gridfil.clat_vertices.values)
    gridfil.close()

    clon_vertices = np.where(
        clon_vertices < -180.0, clon_vertices + 360.0, clon_vertices
    )
    clon_vertices = np.where(
        clon_vertices > 180.0, clon_vertices - 360.0, clon_vertices
    )
    triangles = np.stack([clon_vertices, clat_vertices], axis=-1)

    # ---------------------------
    # Shared color range across BOTH panels
    # ---------------------------
    if vmin is None or vmax is None:
        combined = np.concatenate(
            [var_top[np.isfinite(var_top)], var_bottom[np.isfinite(var_bottom)]]
        )
        if percentile_clip:
            p_lo_val, p_hi_val = np.nanpercentile(combined, [p_low, p_high])
            if symmetric:
                vmax_abs = max(abs(p_lo_val), abs(p_hi_val))
                vmin, vmax = -vmax_abs, vmax_abs
            else:
                vmin, vmax = p_lo_val, p_hi_val
        else:
            vmin, vmax = np.nanmin(combined), np.nanmax(combined)

    print(f"Using shared color range: vmin={vmin:.4f}, vmax={vmax:.4f}")

    cmap_obj = plt.get_cmap(cmap)
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

    # 2 rows, 1 column vertical stack layout (Standard paper width 7.0 in)
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(7.0, 5.8),
        subplot_kw=dict(projection=ccrs.PlateCarree()),
        constrained_layout=True,
    )

    datasets = [
        (var_top, title_top, axes[0]),
        (var_bottom, title_bottom, axes[1]),
    ]

    for var, subtitle, ax in datasets:
        colors = cmap_obj(norm(var))
        colors[~np.isfinite(var)] = [1, 1, 1, 0]

        ax.set_global()
        ax.coastlines(linewidth=0.6, zorder=2)

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.4,
            color="dimgray",
            alpha=0.45,
            zorder=2,
        )
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {"size": fontsize - 2}
        gl.ylabel_style = {"size": fontsize - 2}

        coll = PolyCollection(
            triangles,
            fc=colors,
            edgecolors="none",
            linewidth=0.05,
            transform=ccrs.Geodetic(),
            zorder=0,
        )
        ax.add_collection(coll)

        # Unbolded subpanel title
        ax.set_title(subtitle, fontsize=fontsize, pad=4)

    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])

    # Vertical colorbar on the right with compact width and padding
    cbar = fig.colorbar(
        sm,
        ax=axes,
        orientation="vertical",
        fraction=0.03,
        pad=0.03,
        shrink=0.85,
        extend="both",
    )
    cbar.set_label(cbar_label, fontsize=fontsize, fontweight="normal")
    cbar.ax.tick_params(labelsize=fontsize - 1)

    os.makedirs(save_dir, exist_ok=True)
    outpath_png = os.path.join(save_dir, f"{filename}.png")
    outpath_pdf = os.path.join(save_dir, f"{filename}.pdf")

    plt.savefig(outpath_png, dpi=400, bbox_inches="tight")
    plt.savefig(outpath_pdf, dpi=400, bbox_inches="tight")
    plt.show()
    print(f"--> saved to {outpath_png} and {outpath_pdf}")


# --- Usage: compute all 8 arrays first ---
uw_test_jan = np.mean(
    metrics_data["uw_test"][:, :22] * 1000, axis=1
)  # Pa -> mPa
uw_test_aug  = np.mean(metrics_data["uw_test"][:, 22:] * 1000, axis=1)
vw_test_jan  = np.mean(metrics_data["vw_test"][:, :22] * 1000, axis=1)
vw_test_aug  = np.mean(metrics_data["vw_test"][:, 22:] * 1000, axis=1)
uw_model_jan = np.mean(metrics_data["uw_model"][:, :22] * 1000, axis=1)
uw_model_aug = np.mean(metrics_data["uw_model"][:, 22:] * 1000, axis=1)
vw_model_jan = np.mean(metrics_data["vw_model"][:, :22] * 1000, axis=1)
vw_model_aug = np.mean(metrics_data["vw_model"][:, 22:] * 1000, axis=1)

# --- uw: January ---
plot_global_map_flux_pair(
    uw_test_jan,
    uw_model_jan,
    save_dir="/work/bb1097/b383521/plots/paper_plots/",
    filename="uw_momentum_flux_mpa_jan",
    title_top="True (January)",
    title_bottom="Predicted (January)",
    cbar_label=r"$\rho\overline{u'w'}$ [mPa]",
    symmetric=True,
)

# --- uw: August ---
plot_global_map_flux_pair(
    uw_test_aug,
    uw_model_aug,
    save_dir="/work/bb1097/b383521/plots/paper_plots/",
    filename="uw_momentum_flux_mpa_aug",
    title_top="True (August)",
    title_bottom="Predicted (August)",
    cbar_label=r"$\rho\overline{u'w'}$ [mPa]",
    symmetric=True,
)

# --- vw: January ---
plot_global_map_flux_pair(
    vw_test_jan,
    vw_model_jan,
    save_dir="/work/bb1097/b383521/plots/paper_plots/",
    filename="vw_momentum_flux_mpa_jan",
    title_top="True (January)",
    title_bottom="Predicted (January)",
    cbar_label=r"$\rho\overline{v'w'}$ [mPa]",
    symmetric=True,
)

# --- vw: August ---
plot_global_map_flux_pair(
    vw_test_aug,
    vw_model_aug,
    save_dir="/work/bb1097/b383521/plots/paper_plots/",
    filename="vw_momentum_flux_mpa_aug",
    title_top="True (August)",
    title_bottom="Predicted (August)",
    cbar_label=r"$\rho\overline{v'w'}$ [mPa]",
    symmetric=True,
)

### compute correlation and bias for uw and vw in Jan and Aug
r_uw_jan = np.corrcoef(uw_test_jan, uw_model_jan)[0, 1]
r_uw_aug = np.corrcoef(uw_test_aug, uw_model_aug)[0, 1]
r_vw_jan = np.corrcoef(vw_test_jan, vw_model_jan)[0, 1]
r_vw_aug = np.corrcoef(vw_test_aug, vw_model_aug)[0, 1]

print(f"uw: January: Spearman r = {r_uw_jan:.3f}, August: Spearman r = {r_uw_aug:.3f}")
print(f"vw: January: Spearman r = {r_vw_jan:.3f}, August: Spearman r = {r_vw_aug:.3f}")

In [ ]:
import os
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.colors import TwoSlopeNorm


def plot_flux_map_jan_aug(
    F_true_jan, F_pred_jan, F_true_aug, F_pred_aug,
    save_dir,
    filename,
    cbar_label,
    cmap="RdBu_r",
    p_clip=98,            # symmetric colour limit from this percentile of |F|
    fontsize=10,
):
    gridfil = xr.open_dataset("/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc")
    lon = np.rad2deg(gridfil.clon_vertices.values)
    lat = np.rad2deg(gridfil.clat_vertices.values)
    gridfil.close()
    lon = np.where(lon < -180, lon + 360, lon)
    lon = np.where(lon > 180, lon - 360, lon)
    tris = np.stack([lon, lat], axis=-1)

    months = {
        "January": (np.asarray(F_true_jan, float), np.asarray(F_pred_jan, float)),
        "August": (np.asarray(F_true_aug, float), np.asarray(F_pred_aug, float)),
    }

    fig, axes = plt.subplots(
        2, 2, figsize=(7.2, 4.8),
        subplot_kw=dict(projection=ccrs.PlateCarree()),
        constrained_layout=True,
    )
    cmo = plt.get_cmap(cmap)

    for col, (month, (ft, fp)) in enumerate(months.items()):
        both = np.concatenate([ft[np.isfinite(ft)], fp[np.isfinite(fp)]])
        vmax = np.percentile(np.abs(both), p_clip)
        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
        print(f"{month}: colour range = ±{vmax:.3f}")

        for row, (var, label) in enumerate([(ft, "True"), (fp, "Predicted")]):
            ax = axes[row, col]
            colors = cmo(norm(var))
            colors[~np.isfinite(var)] = [1, 1, 1, 0]

            ax.set_global()
            ax.add_collection(PolyCollection(
                tris, fc=colors, edgecolors="none",
                transform=ccrs.Geodetic(), zorder=0))
            ax.coastlines(linewidth=0.5, zorder=2)

            gl = ax.gridlines(draw_labels=True, linewidth=0.3,
                              color="dimgray", alpha=0.45, zorder=2)
            gl.top_labels = False
            gl.right_labels = False
            gl.left_labels = (col == 0)
            gl.bottom_labels = (row == 1)
            gl.xlabel_style = gl.ylabel_style = {"size": fontsize - 3}
            ax.set_title(f"{label} ({month})", fontsize=fontsize, pad=3)

        sm = plt.cm.ScalarMappable(cmap=cmo, norm=norm)
        sm.set_array([])
        cb = fig.colorbar(sm, ax=axes[:, col], orientation="horizontal",
                          fraction=0.06, pad=0.04, shrink=0.9, extend="both")
        cb.set_label(cbar_label, fontsize=fontsize)
        cb.ax.tick_params(labelsize=fontsize - 1)

    os.makedirs(save_dir, exist_ok=True)
    for ext in ("png", "pdf"):
        plt.savefig(os.path.join(save_dir, f"{filename}.{ext}"),
                    dpi=400, bbox_inches="tight")
    plt.show()


# --- data (Pa -> mPa), time-averaged per month ---
uw_test_jan  = np.mean(metrics_data["uw_test"][:, :22]  * 1000, axis=1)
uw_test_aug  = np.mean(metrics_data["uw_test"][:, 22:]  * 1000, axis=1)
uw_model_jan = np.mean(metrics_data["uw_model"][:, :22] * 1000, axis=1)
uw_model_aug = np.mean(metrics_data["uw_model"][:, 22:] * 1000, axis=1)
vw_test_jan  = np.mean(metrics_data["vw_test"][:, :22]  * 1000, axis=1)
vw_test_aug  = np.mean(metrics_data["vw_test"][:, 22:]  * 1000, axis=1)
vw_model_jan = np.mean(metrics_data["vw_model"][:, :22] * 1000, axis=1)
vw_model_aug = np.mean(metrics_data["vw_model"][:, 22:] * 1000, axis=1)

save_dir = "/work/bb1097/b383521/plots/paper_plots/"

# --- zonal flux ---
plot_flux_map_jan_aug(
    uw_test_jan, uw_model_jan, uw_test_aug, uw_model_aug,
    save_dir, "uw_momentum_flux_jan_aug",
    cbar_label=r"$\rho\overline{u'w'}$ (mPa)",
)

# --- meridional flux ---
plot_flux_map_jan_aug(
    vw_test_jan, vw_model_jan, vw_test_aug, vw_model_aug,
    save_dir, "vw_momentum_flux_jan_aug",
    cbar_label=r"$\rho\overline{v'w'}$ (mPa)",
)

# --- correlations (Pearson, as in the table) ---
for name, (tj, mj, ta, ma) in {
    "uw": (uw_test_jan, uw_model_jan, uw_test_aug, uw_model_aug),
    "vw": (vw_test_jan, vw_model_jan, vw_test_aug, vw_model_aug),
}.items():
    print(f"{name}: Jan r = {np.corrcoef(tj, mj)[0, 1]:.3f}, "
          f"Aug r = {np.corrcoef(ta, ma)[0, 1]:.3f}")

In [ ]:
import os
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.colors import SymLogNorm, TwoSlopeNorm


def plot_flux_map_jan_aug(
    F_true_jan, F_pred_jan, F_true_aug, F_pred_aug,
    save_dir,
    filename,
    cbar_label,
    cmap="RdBu_r",
    p_clip=99,              # symmetric colour limit from this percentile of |F|
    scale="symlog",         # "symlog" or "linear"
    linthresh_frac=0.08,    # linear region = +-linthresh_frac * vmax (symlog only)
    fontsize=10,
):
    gridfil = xr.open_dataset("/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc")
    lon = np.rad2deg(gridfil.clon_vertices.values)
    lat = np.rad2deg(gridfil.clat_vertices.values)
    gridfil.close()
    lon = np.where(lon < -180, lon + 360, lon)
    lon = np.where(lon > 180, lon - 360, lon)
    tris = np.stack([lon, lat], axis=-1)

    months = {
        "January": (np.asarray(F_true_jan, float), np.asarray(F_pred_jan, float)),
        "August": (np.asarray(F_true_aug, float), np.asarray(F_pred_aug, float)),
    }

    fig, axes = plt.subplots(
        2, 2, figsize=(7.2, 4.8),
        subplot_kw=dict(projection=ccrs.PlateCarree()),
        constrained_layout=True,
    )
    cmo = plt.get_cmap(cmap)

    for col, (month, (ft, fp)) in enumerate(months.items()):
        both = np.concatenate([ft[np.isfinite(ft)], fp[np.isfinite(fp)]])
        vmax = np.percentile(np.abs(both), p_clip)

        if scale == "symlog":
            linthresh = linthresh_frac * vmax
            norm = SymLogNorm(linthresh=linthresh, linscale=0.7,
                              vmin=-vmax, vmax=vmax, base=10)
            print(f"{month}: range +-{vmax:.3f} mPa, linthresh {linthresh:.3f} mPa")
        else:
            norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
            print(f"{month}: range +-{vmax:.3f} mPa (linear)")

        for row, (var, label) in enumerate([(ft, "True"), (fp, "Predicted")]):
            ax = axes[row, col]
            colors = cmo(norm(var))
            colors[~np.isfinite(var)] = [1, 1, 1, 0]

            ax.set_global()
            ax.add_collection(PolyCollection(
                tris, fc=colors, edgecolors="none",
                transform=ccrs.Geodetic(), zorder=0))
            ax.coastlines(linewidth=0.5, zorder=2)

            gl = ax.gridlines(draw_labels=True, linewidth=0.3,
                              color="dimgray", alpha=0.45, zorder=2)
            gl.top_labels = False
            gl.right_labels = False
            gl.left_labels = (col == 0)
            gl.bottom_labels = (row == 1)
            gl.xlabel_style = gl.ylabel_style = {"size": fontsize - 3}
            ax.set_title(f"{label} ({month})", fontsize=fontsize, pad=3)

        sm = plt.cm.ScalarMappable(cmap=cmo, norm=norm)
        sm.set_array([])
        cb = fig.colorbar(sm, ax=axes[:, col], orientation="horizontal",
                          fraction=0.06, pad=0.04, shrink=0.9, extend="both")
        if scale == "symlog":
            t = np.array([0.1, 0.3, 1.0]) * vmax
            ticks = np.concatenate([-t[::-1], [0], t])
            cb.set_ticks(ticks)
            cb.set_ticklabels([f"{v:.2g}" for v in ticks])
        cb.set_label(cbar_label, fontsize=fontsize)
        cb.ax.tick_params(labelsize=fontsize - 1)

    os.makedirs(save_dir, exist_ok=True)
    for ext in ("png", "pdf"):
        plt.savefig(os.path.join(save_dir, f"{filename}.{ext}"),
                    dpi=400, bbox_inches="tight")
    plt.show()


# --- data (Pa -> mPa), time-averaged per month ---
uw_test_jan  = np.mean(metrics_data["uw_test"][:, :22]  * 1000, axis=1)
uw_test_aug  = np.mean(metrics_data["uw_test"][:, 22:]  * 1000, axis=1)
uw_model_jan = np.mean(metrics_data["uw_model"][:, :22] * 1000, axis=1)
uw_model_aug = np.mean(metrics_data["uw_model"][:, 22:] * 1000, axis=1)
vw_test_jan  = np.mean(metrics_data["vw_test"][:, :22]  * 1000, axis=1)
vw_test_aug  = np.mean(metrics_data["vw_test"][:, 22:]  * 1000, axis=1)
vw_model_jan = np.mean(metrics_data["vw_model"][:, :22] * 1000, axis=1)
vw_model_aug = np.mean(metrics_data["vw_model"][:, 22:] * 1000, axis=1)

save_dir = "/work/bb1097/b383521/plots/paper_plots/"

# --- zonal flux ---
plot_flux_map_jan_aug(
    uw_test_jan, uw_model_jan, uw_test_aug, uw_model_aug,
    save_dir, "uw_momentum_flux_jan_aug",
    cbar_label=r"$\rho\overline{u'w'}$ (mPa)",
)

# --- meridional flux ---
plot_flux_map_jan_aug(
    vw_test_jan, vw_model_jan, vw_test_aug, vw_model_aug,
    save_dir, "vw_momentum_flux_jan_aug",
    cbar_label=r"$\rho\overline{v'w'}$ (mPa)",
)

# --- correlations (Pearson, same as in the table) ---
for name, (tj, mj, ta, ma) in {
    "uw": (uw_test_jan, uw_model_jan, uw_test_aug, uw_model_aug),
    "vw": (vw_test_jan, vw_model_jan, vw_test_aug, vw_model_aug),
}.items():
    print(f"{name}: Jan r = {np.corrcoef(tj, mj)[0, 1]:.3f}, "
          f"Aug r = {np.corrcoef(ta, ma)[0, 1]:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_zonal_mean_pair(
    lat,
    test_jan,
    model_jan,
    test_aug,
    model_aug,
    ylabel=r"$\rho\overline{u'w'}$ [mPa]",
    savename="zonal_mean_uw",
    fontsize=11,  # Standard paper font size (9–11 pt)
):
    # Standard 2-column paper figure dimensions
    fig, axes = plt.subplots(
        1, 2, figsize=(7.0, 3.2), sharey=True, constrained_layout=True
    )

    datasets = [
        ("January", test_jan, model_jan, axes[0]),
        ("August", test_aug, model_aug, axes[1]),
    ]

    handles = []

    for name, true_vals, pred_vals, ax in datasets:
        ax.axhline(0, color="gray", lw=0.8, ls=":", alpha=0.7)

        (l1,) = ax.plot(lat, true_vals, "-", color="black", lw=1.8, label="True")
        (l2,) = ax.plot(
            lat, pred_vals, "--", color="firebrick", lw=1.8, label="Predicted"
        )

        ax.set_title(name, fontsize=fontsize, pad=6)
        ax.set_xlim(lat.min(), lat.max())

        # Clean degree-formatted latitude ticks
        xticks = np.arange(-90, 91, 30)
        xticklabels = [
            f"{abs(x)}°S" if x < 0 else (f"{x}°N" if x > 0 else "0°")
            for x in xticks
        ]
        ax.set_xticks(xticks)
        ax.set_xticklabels(xticklabels, fontsize=fontsize - 1)

        ax.tick_params(axis="both", labelsize=fontsize - 1)
        ax.grid(alpha=0.25, lw=0.5)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if ax == axes[0]:
            ax.set_ylabel(ylabel, fontsize=fontsize)
            handles = [l1, l2]

    # Centered figure-level x-axis label
    fig.supxlabel("Latitude", fontsize=fontsize)

    # Clean upper legend
    fig.legend(
        handles=handles,
        labels=["True", "Predicted"],
        loc="lower center",
        ncol=2,
        bbox_to_anchor=(0.5, 1.01),
        fontsize=fontsize - 1,
        frameon=False,
    )

    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.png",
        dpi=400,
        bbox_inches="tight",
    )
    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.pdf",
        dpi=400,
        bbox_inches="tight",
    )
    plt.show()


# --- Usage ---
uw_test_zonal_mean_jan = np.mean(
    metrics_data["uw_test_zonal_mean"][:, :22] * 1000, axis=1
)  # Pa -> mPa
uw_test_zonal_mean_aug = np.mean(
    metrics_data["uw_test_zonal_mean"][:, 22:] * 1000, axis=1
)
uw_model_zonal_mean_jan = np.mean(
    metrics_data["uw_model_zonal_mean"][:, :22] * 1000, axis=1
)
uw_model_zonal_mean_aug = np.mean(
    metrics_data["uw_model_zonal_mean"][:, 22:] * 1000, axis=1
)

vw_test_zonal_mean_jan = np.mean(
    metrics_data["vw_test_zonal_mean"][:, :22] * 1000, axis=1
)  # Pa -> mPa
vw_test_zonal_mean_aug = np.mean(
    metrics_data["vw_test_zonal_mean"][:, 22:] * 1000, axis=1
)
vw_model_zonal_mean_jan = np.mean(
    metrics_data["vw_model_zonal_mean"][:, :22] * 1000, axis=1
)
vw_model_zonal_mean_aug = np.mean(
    metrics_data["vw_model_zonal_mean"][:, 22:] * 1000, axis=1
)

lat_bins = metrics_data["lat_bins"]
lat = 0.5 * (lat_bins[:-1] + lat_bins[1:])

plot_zonal_mean_pair(
    lat,
    uw_test_zonal_mean_jan,
    uw_model_zonal_mean_jan,
    uw_test_zonal_mean_aug,
    uw_model_zonal_mean_aug,
    ylabel=r"$\rho\overline{u'w'}$ [mPa]",
    savename="zonal_mean_uw",
)

plot_zonal_mean_pair(
    lat,
    vw_test_zonal_mean_jan,
    vw_model_zonal_mean_jan,
    vw_test_zonal_mean_aug,
    vw_model_zonal_mean_aug,
    ylabel=r"$\rho\overline{v'w'}$ [mPa]",
    savename="zonal_mean_vw",
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_zonal_mean_four(
    lat,
    uw_test_jan, uw_model_jan, uw_test_aug, uw_model_aug,
    vw_test_jan, vw_model_jan, vw_test_aug, vw_model_aug,
    savename="zonal_mean_uw_vw",
    layout="1x4",          # "1x4" (one row) or "2x2" (rows: uw, vw)
    show_r=True,           # correlation of the two profiles in each panel
    fontsize=10,
):
    uw_label = r"$\rho\overline{u'w'}$ (mPa)"
    vw_label = r"$\rho\overline{v'w'}$ (mPa)"

    # (flux label, month, true, predicted)
    panels = [
        (uw_label, "January", uw_test_jan, uw_model_jan),
        (uw_label, "August",  uw_test_aug, uw_model_aug),
        (vw_label, "January", vw_test_jan, vw_model_jan),
        (vw_label, "August",  vw_test_aug, vw_model_aug),
    ]

    if layout == "2x2":
        fig, axes = plt.subplots(2, 2, figsize=(5.6, 6.0), sharey=True,
                                 constrained_layout=True)
        axes = [axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]]
    else:
        fig, axes = plt.subplots(1, 4, figsize=(7.2, 3.4), sharey=True,
                                 constrained_layout=True)

    # shared x-range per flux (January and August comparable), symmetric about 0
    xlim = {}
    for label in (uw_label, vw_label):
        vals = np.concatenate([np.concatenate([t, p])
                               for lab, _, t, p in panels if lab == label])
        m = 1.08 * np.nanmax(np.abs(vals))
        xlim[label] = (-m, m)

    yticks = np.arange(-90, 91, 30)
    yticklabels = [f"{abs(y)}°S" if y < 0 else (f"{y}°N" if y > 0 else "0°")
                   for y in yticks]

    handles = None
    for i, (ax, (label, month, tr, pr)) in enumerate(zip(axes, panels)):
        ax.axvline(0, color="gray", lw=0.8, ls=":", alpha=0.8)
        ax.fill_betweenx(lat, tr, pr, color="firebrick", alpha=0.12, lw=0)
        (l1,) = ax.plot(tr, lat, "-", color="black", lw=1.6, label="True")
        (l2,) = ax.plot(pr, lat, "--", color="firebrick", lw=1.6, label="Predicted")
        handles = [l1, l2]

        ax.set_title(month, fontsize=fontsize, pad=4)
        ax.set_xlim(*xlim[label])
        ax.set_ylim(lat.min(), lat.max())
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticklabels)
        ax.tick_params(labelsize=fontsize - 1)
        ax.grid(alpha=0.25, lw=0.5)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_xlabel(label, fontsize=fontsize)

        if show_r:
            r = np.corrcoef(tr, pr)[0, 1]
            ax.text(0.04, 0.03, f"$r={r:.2f}$", transform=ax.transAxes,
                    fontsize=fontsize - 2, ha="left", va="bottom",
                    bbox=dict(fc="w", ec="none", alpha=0.8, pad=1.5))

        first_in_row = (i == 0) if layout != "2x2" else (i % 2 == 0)
        if first_in_row:
            ax.set_ylabel("Latitude", fontsize=fontsize)

    fig.legend(handles=handles, labels=["True", "Predicted"], loc="lower center",
               ncol=2, bbox_to_anchor=(0.5, 1.0), fontsize=fontsize - 1,
               frameon=False)

    for ext in ("png", "pdf"):
        plt.savefig(f"/work/bb1097/b383521/plots/paper_plots/{savename}.{ext}",
                    dpi=400, bbox_inches="tight")
    plt.show()


# --- data (Pa -> mPa), time-averaged per month ---
def zm(key, sl):
    return np.mean(metrics_data[key][:, sl] * 1000, axis=1)

jan, aug = slice(0, 22), slice(22, None)

lat_bins = metrics_data["lat_bins"]
lat = 0.5 * (lat_bins[:-1] + lat_bins[1:])

plot_zonal_mean_four(
    lat,
    zm("uw_test_zonal_mean", jan), zm("uw_model_zonal_mean", jan),
    zm("uw_test_zonal_mean", aug), zm("uw_model_zonal_mean", aug),
    zm("vw_test_zonal_mean", jan), zm("vw_model_zonal_mean", jan),
    zm("vw_test_zonal_mean", aug), zm("vw_model_zonal_mean", aug),
    savename="zonal_mean_uw_vw",
    layout="1x4",
)

In [ ]:
import numpy as np

# Zonal (u'w') - January and August
for name, true_arr, pred_arr in [
    ("uw January", uw_test_zonal_mean_jan, uw_model_zonal_mean_jan),
    ("uw August", uw_test_zonal_mean_aug, uw_model_zonal_mean_aug),
]:
    idx_min = np.argmin(true_arr)
    idx_max = np.argmax(true_arr)
    print(f"{name}: true min={true_arr[idx_min]:.3f} at lat={lat[idx_min]:.1f}, "
          f"pred at same lat={pred_arr[idx_min]:.3f}, "
          f"underprediction={(1-pred_arr[idx_min]/true_arr[idx_min])*100:.1f}%")
    print(f"{name}: true max={true_arr[idx_max]:.3f} at lat={lat[idx_max]:.1f}, "
          f"pred at same lat={pred_arr[idx_max]:.3f}, "
          f"underprediction={(1-pred_arr[idx_max]/true_arr[idx_max])*100:.1f}%")

# Meridional (v'w') - January and August
for name, true_arr, pred_arr in [
    ("vw January", vw_test_zonal_mean_jan, vw_model_zonal_mean_jan),
    ("vw August", vw_test_zonal_mean_aug, vw_model_zonal_mean_aug),
]:
    idx_max = np.argmax(true_arr)
    print(f"{name}: true peak={true_arr[idx_max]:.3f} at lat={lat[idx_max]:.1f}, "
          f"pred at same lat={pred_arr[idx_max]:.3f}, "
          f"underprediction={(1-pred_arr[idx_max]/true_arr[idx_max])*100:.1f}%")

window_mask = (lat >= 28) & (lat <= 34)
print("True uw in window:", uw_test_zonal_mean_jan[window_mask])
print("Pred uw in window:", uw_model_zonal_mean_jan[window_mask])

In [ ]:
for name, t, p in [("uw Jan", uw_test_zonal_mean_jan, uw_model_zonal_mean_jan),
                   ("uw Aug", uw_test_zonal_mean_aug, uw_model_zonal_mean_aug),
                   ("vw Jan", vw_test_zonal_mean_jan, vw_model_zonal_mean_jan),
                   ("vw Aug", vw_test_zonal_mean_aug, vw_model_zonal_mean_aug)]:
    # each extremum of the true profile (largest positive and largest negative)
    for k, label in [(np.argmax(t), "max"), (np.argmin(t), "min")]:
        print(f"{name} {label}: lat={lat[k]:.1f}  true={t[k]:.3f}  pred={p[k]:.3f}  "
              f"underestimate={100*(1-p[k]/t[k]):.0f}%")

# sign check, vw in January, 28-34N
m = (lat > 28) & (lat < 34)
print(np.c_[lat[m], vw_test_zonal_mean_jan[m], vw_model_zonal_mean_jan[m]])

In [ ]:
import numpy as np
import xarray as xr
gridfil = xr.open_dataset("/work/bb1097/b381585/data/icon_grid_0012_R02B04_G.nc")
clat_vertices = np.rad2deg(gridfil.clat_vertices.values)  # shape (n_triangles, 3)
lat_triangle = clat_vertices.mean(axis=1)  # centroid latitude per triangle
# You should already have per-triangle latitude coordinates available
# (e.g., from the same grid file used in plot_global_map / plot_global_map_flux)
# lat_triangle: shape (n_triangles,) -- latitude of each coarse-grid triangle centroid

lat_mask = (lat_triangle >= 28) & (lat_triangle <= 34)

uw_true_band = uw_test_jan[lat_mask]
uw_pred_band = uw_model_jan[lat_mask]

print(f"N triangles in 28-34N band: {lat_mask.sum()}")

print(f"\nTrue uw at 28-34N: min={uw_true_band.min():.3f}, max={uw_true_band.max():.3f}, "
      f"mean={uw_true_band.mean():.3f}")
print(f"Fraction of positive true values: {(uw_true_band > 0).mean():.3f}")

print(f"\nPred uw at 28-34N: min={uw_pred_band.min():.3f}, max={uw_pred_band.max():.3f}, "
      f"mean={uw_pred_band.mean():.3f}")
print(f"Fraction of positive pred values: {(uw_pred_band > 0).mean():.3f}")

# Check whether the true positive mean is dominated by a small number of large values
pos_true = uw_true_band[uw_true_band > 0]
sorted_pos_true = np.sort(pos_true)[::-1]
print(f"\nN positive true values: {len(pos_true)} out of {len(uw_true_band)} total")
print(f"Top 5 true positive values (descending): {sorted_pos_true[:5]}")
print(f"Top 1 value as fraction of total positive sum: {sorted_pos_true[0] / pos_true.sum():.3f}")
print(f"Top 5 values as fraction of total positive sum: {sorted_pos_true[:5].sum() / pos_true.sum():.3f}")

# Same diagnostic on the model side, for comparison
pos_pred = uw_pred_band[uw_pred_band > 0]
if len(pos_pred) > 0:
    sorted_pos_pred = np.sort(pos_pred)[::-1]
    print(f"\nN positive PRED values: {len(pos_pred)} out of {len(uw_pred_band)} total")
    print(f"Top 5 pred positive values (descending): {sorted_pos_pred[:5]}")
else:
    print("\nModel predicts NO positive values in this latitude band.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

shap_dir = "/work/bb1097/b383521/ML_models/coarsed_grained/shap/"

sv_ocean = np.load(shap_dir + "svglobal_ocean_2.npy")  # (100, 102, 144)
sv_land  = np.load(shap_dir + "svglobal_land_2.npy")   # (100, 111, 144)

keys = ["u", "v", "w", "p", "T", "N", "b", "Tw", "dxu", "dyu", "dxv",
        "dyv", "dxT", "dyT", "dxp", "dyp", "vorticity"]
n_keys = len(keys)  # 17
n_dynamic = n_keys * 6  # 102

# Land-only orography/mask feature names -- UPDATE these to match your actual 9 (or 10) features
orography_keys = [
    "H_var",        # 0: total orographic variance (roughness), H_sum
    "mean_k",       # 1: power-weighted mean zonal wavenumber
    "mean_l",       # 2: power-weighted mean meridional wavenumber (signed)
    "ridge_wlen_k", # 3: effective zonal wavelength (1/k, weighted)
    "ridge_wlen_l", # 4: effective meridional wavelength (1/|l|, weighted)
    "spec_width_k", # 5: zonal spectral width
    "spec_width_l", # 6: meridional spectral width
    "anisotropy",   # 7: zonal/meridional power ratio
    "orientation",  # 8: dominant ridge orientation angle
]# adjust names/order to match your actual feature construction

def compute_importance(sv, has_orography=False):
    importance_full = np.mean(np.abs(sv), axis=(0, 2))  # (n_features,)
    dynamic_part = importance_full[:n_dynamic]
    importance_by_level = dynamic_part.reshape(n_keys, 6)
    importance_per_variable = importance_by_level.sum(axis=1)  # (17,)

    result = {"per_variable": importance_per_variable, "by_level": importance_by_level}
    if has_orography:
        result["orography"] = importance_full[n_dynamic:]
    return result

ocean_imp = compute_importance(sv_ocean, has_orography=False)
land_imp  = compute_importance(sv_land, has_orography=True)

# --- Combine dynamical + orography for land into one full set of names/values for plotting ---
land_all_names = keys + orography_keys[:len(land_imp["orography"])]
land_all_values = np.concatenate([land_imp["per_variable"], land_imp["orography"]])

ocean_all_names = keys
ocean_all_values = ocean_imp["per_variable"]

groups = {
    "Dynamical (u, v)": ["u", "v"],
    "Thermodynamic (b, T, N)": ["b", "T", "N"],
    "Pressure (p)": ["p"],
    "Vertical motion (w)": ["w"],
    "Flux/other (Tw, vorticity)": ["Tw", "vorticity"],
    "Gradients": ["dxu","dyu","dxv","dyv","dxT","dyT","dxp","dyp"],
}

def compute_grouped_importance(importance_per_variable, keys, groups):
    grouped = {}
    for group_name, var_list in groups.items():
        idx = [keys.index(v) for v in var_list]
        grouped[group_name] = importance_per_variable[idx].sum()
    return grouped

ocean_grouped = compute_grouped_importance(ocean_imp["per_variable"], keys, groups)
land_grouped  = compute_grouped_importance(land_imp["per_variable"], keys, groups)

# Land also has orography as its own group
land_grouped["Orography (land only)"] = land_imp["orography"].sum()

print("=== OCEAN (grouped) ===")
for k, v in sorted(ocean_grouped.items(), key=lambda x: -x[1]):
    print(f"{k:<28}: {v:.4f}")

print("\n=== LAND (grouped) ===")
for k, v in sorted(land_grouped.items(), key=lambda x: -x[1]):
    print(f"{k:<28}: {v:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm

keys = ["u", "v", "w", "p", "T", "N", "b", "Tw", "dxu", "dyu", "dxv",
        "dyv", "dxT", "dyT", "dxp", "dyp", "vorticity"]
labels = [
    r"$u$", r"$v$", r"$w$", r"$p$", r"$T$", r"$N^{2}$", r"$b$",
    r"$\overline{T'w'}$",
    r"$\frac{\partial u}{\partial x}$", r"$\frac{\partial u}{\partial y}$",
    r"$\frac{\partial v}{\partial x}$", r"$\frac{\partial v}{\partial y}$",
    r"$\frac{\partial T}{\partial x}$", r"$\frac{\partial T}{\partial y}$",
    r"$\frac{\partial p}{\partial x}$", r"$\frac{\partial p}{\partial y}$",
    r"$\zeta$",
]
n_keys, n_levels = len(keys), 6
n_dynamic = n_keys * n_levels
heights = [15.113, 13.021, 11.021, 9.021, 7.021, 5.033]

sv_ocean_height = np.mean(np.abs(sv_ocean[:, :n_dynamic, :]), axis=(0, 2))
sv_land_height  = np.mean(np.abs(sv_land[:, :n_dynamic, :]),  axis=(0, 2))

ocean_map = sv_ocean_height.reshape(n_keys, n_levels).T
land_map  = sv_land_height.reshape(n_keys, n_levels).T

vmax = max(ocean_map.max(), land_map.max())
norm = PowerNorm(gamma=0.4, vmin=0, vmax=vmax)   # lower gamma = more boost for small values

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True,
                         constrained_layout=True)

for ax, data, title in zip(axes, [ocean_map, land_map], ["Ocean", "Land"]):
    im = ax.imshow(data, aspect="auto", origin="upper",
                   cmap="YlOrRd", norm=norm, interpolation="nearest")
    ax.set_title(title)
    ax.set_xticks(np.arange(n_keys))
    ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=10)
    ax.set_yticks(np.arange(n_levels))
    ax.set_yticklabels([f"{h:.1f}" for h in heights])
    ax.set_xlabel("Input variable")

    # thin separators between groups: state vars | flux | gradients | vorticity
    for x in (6.5, 7.5, 15.5):
        ax.axvline(x, color="k", lw=0.8, alpha=0.5)

axes[0].set_ylabel("Height [km]")

cbar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("Mean |SHAP|")
cbar.set_ticks([0, 0.01, 0.05, 0.1, 0.2, 0.4, 0.6])   # denser ticks at the low end
cbar.ax.set_yticklabels(["0", "0.01", "0.05", "0.1", "0.2", "0.4", "0.6"])
plt.savefig("/work/bb1097/b383521/plots/paper_plots/SHAP_heights.png")
plt.show()

In [ ]:
import numpy as np

keys = ["u", "v", "w", "p", "T", "N", "b", "Tw", "dxu", "dyu", "dxv",
        "dyv", "dxT", "dyT", "dxp", "dyp", "vorticity"]
n_keys, n_levels = len(keys), 6
n_dynamic = n_keys * n_levels
heights = [15.113, 13.021, 11.021, 9.021, 7.021, 5.033]  # km

sv_ocean_height = np.mean(np.abs(sv_ocean[:, :n_dynamic, :]), axis=(0, 2))
sv_land_height  = np.mean(np.abs(sv_land[:, :n_dynamic, :]),  axis=(0, 2))

# (17 variables, 6 heights): rows = variables, columns = heights
ocean_tab = sv_ocean_height.reshape(n_keys, n_levels)
land_tab  = sv_land_height.reshape(n_keys, n_levels)


def format_table(tab, title):
    col_w = 12
    lines = [title, "-" * (12 + col_w * n_levels)]
    header = f"{'variable':<12}" + "".join(f"{h:>{col_w}.3f}" for h in heights)
    lines.append(header + "   <- height [km]")
    lines.append("-" * (12 + col_w * n_levels))
    for key, row in zip(keys, tab):
        lines.append(f"{key:<12}" + "".join(f"{v:>{col_w}.4e}" for v in row))
    lines.append("")
    return "\n".join(lines)


out_path = "shap_mean_abs_by_variable_height.txt"
with open(out_path, "w") as f:
    f.write("Mean |SHAP| per input variable and height\n")
    f.write("(averaged over samples and output bins)\n\n")
    f.write(format_table(ocean_tab, "OCEAN"))
    f.write("\n")
    f.write(format_table(land_tab, "LAND"))

print(open(out_path).read())

In [ ]:
for i in range(4):
    print(f"land avarge of the thermodynmic at vertical wave length of {2*np.pi/m[i]/1000}km attribution {np.mean(pct_land[:, :, i])}")
    print(f"ocean avarge of the thermodynmic attribution at vertical wave length of {2*np.pi/m[i]/1000}km {np.mean(pct_ocean[:, :, i])}")
    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def compute_group_spectral_map(sv, keys, group_vars, n_k=6, n_l=6, n_m=4):
    """
    Returns (6,6,4) map of summed |SHAP| for the given group,
    averaged over samples, summed over the group's variables and their 6 levels.
    """
    group_shap_sum = np.zeros((sv.shape[0], sv.shape[2]))  # (n_samples, 144)
    for var in group_vars:
        var_start = keys.index(var) * 6
        var_shap = sv[:, var_start:var_start+6, :]           # (n_samples, 6, 144)
        group_shap_sum += np.sum(np.abs(var_shap), axis=1)   # sum over levels -> (n_samples, 144)
    group_shap_mean = np.mean(group_shap_sum, axis=0)         # avg over samples -> (144,)
    return group_shap_mean.reshape(n_k, n_l, n_m)


def compute_relative_attribution_map(sv, keys, group_a_vars, group_b_vars):
    """
    Returns (6,6,4) map of group_a's percentage share relative to (group_a + group_b),
    per spectral bin: 100 * I_a / (I_a + I_b)
    """
    map_a = compute_group_spectral_map(sv, keys, group_a_vars)
    map_b = compute_group_spectral_map(sv, keys, group_b_vars)
    pct_a = 100.0 * map_a / (map_a + map_b)
    return pct_a


def plot_relative_attribution_ocean_land(
    field_ocean,
    field_land,
    k,
    l,
    m,
    cbar_label,
    savename,
    cmap="RdBu_r",
    vmin=0,
    vmax=100,
    fontsize=11,
    n_ticks=3,
):
    field_ocean = np.nan_to_num(field_ocean, nan=0.0)
    field_land = np.nan_to_num(field_land, nan=0.0)

    extent = [k.min(), k.max(), l.min(), l.max()]
    nm = len(m)

    fig, axes = plt.subplots(
        2,
        nm,
        figsize=(7.0, 3.8),
        sharex="col",
        sharey="row",
        constrained_layout=True,
    )

    if nm == 1:
        axes = axes.reshape(2, 1)

    for j in range(nm):
        lambda_km = 2 * np.pi / np.abs(m[j]) / 1000.0
        fields = [field_ocean[:, :, j], field_land[:, :, j]]

        for i in range(2):
            ax = axes[i, j]

            im = ax.imshow(
                fields[i].T,
                origin="lower",
                extent=extent,
                interpolation="nearest",
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
            )

            ax.axhline(0, color="k", lw=0.6, alpha=0.4)
            ax.axvline(0, color="k", lw=0.6, alpha=0.4)

            if i == 0:
                ax.set_title(
                    rf"$\lambda_z = {lambda_km:.1f}\text{{ km}}$",
                    fontsize=fontsize,
                    pad=6,
                )

            if j == 0:
                label_prefix = "Ocean" if i == 0 else "Land"
                ax.set_ylabel(label_prefix, fontsize=fontsize, labelpad=8)

            ax.xaxis.set_major_locator(mticker.MaxNLocator(n_ticks))
            ax.yaxis.set_major_locator(mticker.MaxNLocator(n_ticks))

            ax.xaxis.set_major_formatter(
                mticker.FuncFormatter(lambda x, pos: f"{int(round(x*1e4))}")
            )
            ax.yaxis.set_major_formatter(
                mticker.FuncFormatter(lambda y, pos: f"{int(round(y*1e4))}")
            )

            ax.tick_params(labelsize=fontsize - 1)

    fig.supxlabel(r"$k \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize)
    fig.supylabel(r"$\ell \quad [10^{-4}\text{ m}^{-1}]$", fontsize=fontsize)

    cbar = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
    cbar.set_label(cbar_label, fontsize=fontsize, fontweight="normal")
    cbar.ax.tick_params(labelsize=fontsize - 1)

    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.pdf",
        dpi=400,
        bbox_inches="tight",
    )
    plt.savefig(
        f"/work/bb1097/b383521/plots/paper_plots/{savename}.png",
        dpi=400,
        bbox_inches="tight",
    )
    plt.show()


# --- Usage ---
pct_ocean = compute_relative_attribution_map(sv_ocean, keys, ["u", "v"], ["b", "T", "N"])
pct_land  = compute_relative_attribution_map(sv_land, keys, ["u", "v"], ["b", "T", "N"])

plot_relative_attribution_ocean_land(
    field_ocean=pct_ocean,
    field_land=pct_land,
    k=k_wavenumber_array,
    l=l_wavenumber_array,
    m=m,
    cbar_label="Momentum share of grouped |SHAP| (%)",
    savename="shap_relative_attribution_thermo",
    cmap="RdBu_r",
)